# 채용 플랫폼 사용자 행동 분석

## 한눈에 보기(개선안)

| 그룹 | 개선 지점 | 제안 |
|---|---|---|
| G0 (신호 없음, 63%) | 지원 시작 전 | 후보 비교판 |
| G1 (단일 신호, 28.6%) | 지원서 작성 중 | 멈춘 단계로 바로 복귀 |
| A 다수 공고 지원형 (151명) | 대량 지원 흐름 | 일괄 확인 후 지원 |
| B 탐색 후 빠른 지원형 (492명) | 지원 직전 | 기업 한눈에 보기 |
| C 다수 공고 빠른 지원형 (171명) | 지원 직후 | 신규 공고 이어주기 |

# 개선안 도출 분석

In [182]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [183]:
final_user_scores

,user_uuid,volume_score,concentration_score,diversity_score,view_score,bookmark_score,volume_null,concentration_null,diversity_null,view_null,bookmark_null,total_score,analysis_group,final_segment
0,2ece8d7d-79b6-4522-928b-8dbae8843fab,1,0,1,0,0,0,0,0,0,0,2,G2_복합신호,A_지원행동복합형
1,24ffac10-a46e-4adb-af66-0074d259c809,0,0,0,0,0,0,0,0,0,0,0,G0_신호없음,G0_신호없음
2,40196d59-ab7c-4951-8a5b-eb87aac0a08b,0,0,1,0,0,0,0,0,0,0,1,G1_단일신호,G1_단일신호
3,3e20b9ee-48bc-46cd-9713-d629f0a51b59,0,0,0,0,0,0,0,0,0,0,0,G0_신호없음,G0_신호없음
4,2bc97037-032e-4137-93d4-feebaa1c2524,0,0,0,0,0,0,0,0,0,0,0,G0_신호없음,G0_신호없음
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10282,bdfebb58-6ebe-4124-8679-de9d5c266dbb,0,0,0,1,0,0,0,0,0,0,1,G1_단일신호,G1_단일신호
10283,dc1fb548-59dd-47bc-82aa-99d92e70f793,0,0,0,0,0,0,0,0,0,0,0,G0_신호없음,G0_신호없음
10284,1e311075-359d-40fd-b606-f74cb1819499,0,0,0,0,0,0,0,0,0,0,0,G0_신호없음,G0_신호없음
10285,7ba43545-bf1c-465e-8996-e55231b020c6,0,0,0,1,0,0,0,0,0,0,1,G1_단일신호,G1_단일신호


In [184]:
import pandas as pd

# path 생성 + timestamp 정리 (한 번만 실행)
log_df['path'] = log_df['URL'].astype(str).str.split('?').str[0].str.strip('/')
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], errors='coerce', utc=True)

n_bad = log_df['timestamp'].isna().sum() + log_df['user_uuid'].isna().sum()
print(f"timestamp/user_uuid 결측 행: {n_bad:,}")
log_df = log_df.dropna(subset=['timestamp', 'user_uuid'])


def build_group_df(seg_prefix, label):
    """
    그룹 사용자의 log_df + app_df + job_df + com_df를 합친 테이블 반환
    (로그 × 지원 곱집합이라 행 수가 큽니다)
    """
    print(f"===== {label} 시작 =====")

    # 1) 그룹 사용자 추출 (사용자당 1행)
    seg_col = 'final_segment' if 'final_segment' in final_user_scores.columns else 'user_group'
    score_cols = ['user_uuid', seg_col, 'total_score', 'volume_score', 'concentration_score',
                  'diversity_score', 'view_score', 'bookmark_score']
    users = final_user_scores[final_user_scores[seg_col].astype(str).str.startswith(seg_prefix)]
    users = users[score_cols].drop_duplicates('user_uuid').copy()
    print(f"[1] {label} 사용자 수: {len(users):,}")

    # 2) 로그 + 점수 (inner)
    df = log_df.merge(users, on='user_uuid', how='inner')
    print(f"[2] 로그 조인 후 행 수: {len(df):,}")

    # 3) 지원 데이터 (left)
    dedup_keys = ['user_uuid', 'job_uuid'] if 'job_uuid' in app_df.columns else ['user_uuid']
    app_small = app_df.drop_duplicates(subset=dedup_keys).copy()
    df = df.merge(app_small, on='user_uuid', how='left', suffixes=('', '_app'))
    print(f"[3] app_df 조인 후 행 수: {len(df):,}")

    # 4) 공고 마스터 (left)
    job_cols = [c for c in ['job_uuid', 'company_uuid', 'job_field', 'career_type_string',
                            'allow_remote', 'can_show_salary', 'start_date', 'end_date']
                if c in job_df.columns]
    job_small = job_df[job_cols].drop_duplicates('job_uuid')
    before = len(df)
    df = df.merge(job_small, on='job_uuid', how='left', suffixes=('', '_job'))
    assert len(df) == before, "job_df 조인에서 행이 늘었습니다 (job_uuid 중복 확인)"
    print(f"[4] job_df 조인 후 행 수: {len(df):,}")

    # 5) 기업 (left)
    com_key = 'company_uuid' if 'company_uuid' in df.columns else 'company_uuid_job'
    com_cols = [c for c in ['company_uuid', 'employee_count', 'view_count',
                            'follow_count', 'reference_count'] if c in com_df.columns]
    com_small = com_df[com_cols].drop_duplicates('company_uuid').rename(columns={'company_uuid': com_key})
    before = len(df)
    df = df.merge(com_small, on=com_key, how='left', suffixes=('', '_com'))
    assert len(df) == before, "com_df 조인에서 행이 늘었습니다 (company_uuid 중복 확인)"
    print(f"[5] com_df 조인 후 행 수: {len(df):,}")

    print(f"===== {label} 완료: {len(df):,}행, 사용자 {df['user_uuid'].nunique():,}명 =====")
    return df


g0_df = build_group_df('G0', 'G0(신호 없음)')
print(list(g0_df.columns))
display(g0_df.head(3))

timestamp/user_uuid 결측 행: 24
===== G0(신호 없음) 시작 =====
[1] G0(신호 없음) 사용자 수: 6,484
[2] 로그 조인 후 행 수: 8,058,510
[3] app_df 조인 후 행 수: 75,047,217
[4] job_df 조인 후 행 수: 75,047,217
[5] com_df 조인 후 행 수: 75,047,217
===== G0(신호 없음) 완료: 75,047,217행, 사용자 6,484명 =====
['user_uuid', 'URL', 'timestamp', 'date', 'response_code', 'method', 'timestamp_kst', 'path', 'final_segment', 'total_score', 'volume_score', 'concentration_score', 'diversity_score', 'view_score', 'bookmark_score', 'cdate', 'company_uuid', 'job_uuid', 'application_uuid', 'cdate_kst', 'company_uuid_job', 'job_field', 'career_type_string', 'allow_remote', 'can_show_salary', 'end_date', 'employee_count', 'view_count', 'follow_count', 'reference_count']


,user_uuid,URL,timestamp,date,response_code,method,timestamp_kst,path,final_segment,total_score,volume_score,concentration_score,diversity_score,view_score,bookmark_score,cdate,company_uuid,job_uuid,application_uuid,cdate_kst,company_uuid_job,job_field,career_type_string,allow_remote,can_show_salary,end_date,employee_count,view_count,follow_count,reference_count
0,8a7e70cd-1630-4a8a-8303-0706af3a7908,api/search/language?name=한국어&_=1655915651225,2022-06-22 16:42:48.247454+00:00,2022-06-23,200,GET,2022-06-23 01:42:48.247454+09:00,api/search/language,G0_신호없음,0,0,0,0,0,0,2022-06-22 17:01:05+00:00,ce942711-4bd5-4986-a30a-edc89edf3675,1d51713b-8886-4090-83c7-63881632c773,990f88d1-7dfe-43cf-ae1c-c62e147ccf81,2022-06-23 02:01:05+09:00,ce942711-4bd5-4986-a30a-edc89edf3675,SW 개발,인턴,0.0,0.0,2022-12-20,201-500명,11038.0,221.0,17.0
1,8a7e70cd-1630-4a8a-8303-0706af3a7908,api/signup/form,2022-06-22 16:19:07.770741+00:00,2022-06-23,200,POST,2022-06-23 01:19:07.770741+09:00,api/signup/form,G0_신호없음,0,0,0,0,0,0,2022-06-22 17:01:05+00:00,ce942711-4bd5-4986-a30a-edc89edf3675,1d51713b-8886-4090-83c7-63881632c773,990f88d1-7dfe-43cf-ae1c-c62e147ccf81,2022-06-23 02:01:05+09:00,ce942711-4bd5-4986-a30a-edc89edf3675,SW 개발,인턴,0.0,0.0,2022-12-20,201-500명,11038.0,221.0,17.0
2,8a7e70cd-1630-4a8a-8303-0706af3a7908,api/users/id/template,2022-06-22 16:41:54.449837+00:00,2022-06-23,200,POST,2022-06-23 01:41:54.449837+09:00,api/users/id/template,G0_신호없음,0,0,0,0,0,0,2022-06-22 17:01:05+00:00,ce942711-4bd5-4986-a30a-edc89edf3675,1d51713b-8886-4090-83c7-63881632c773,990f88d1-7dfe-43cf-ae1c-c62e147ccf81,2022-06-23 02:01:05+09:00,ce942711-4bd5-4986-a30a-edc89edf3675,SW 개발,인턴,0.0,0.0,2022-12-20,201-500명,11038.0,221.0,17.0


In [202]:
log_df['path'] = log_df['URL'].astype(str).str.split('?').str[0].str.strip('/')
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], errors='coerce', utc=True)
log_df = log_df.dropna(subset=['timestamp', 'user_uuid'])

# G0 퍼널분석

In [201]:
import pandas as pd

# 1) G0 사용자의 로그만 꺼내기
seg_col = 'final_segment' if 'final_segment' in final_user_scores.columns else 'user_group'
g0_users = final_user_scores.loc[final_user_scores[seg_col].astype(str).str.startswith('G0'), 'user_uuid']
g0_log = log_df[log_df['user_uuid'].isin(g0_users)][['user_uuid', 'timestamp', 'path']].copy()

# 2) 각 단계에 해당하는 로그 (사용자, 시각)만 뽑기
def pick(condition):
    return g0_log.loc[condition, ['user_uuid', 'timestamp']].rename(columns={'timestamp': 't'}).sort_values('t')

path = g0_log['path'].astype(str)
steps = {
    '1. 검색':      pick(path.str.contains('api/search', case=False, na=False)),
    '2. 상세 보기': pick(path == 'api/jobs/job_title'),
    '3. STEP 1':    pick(path.str.contains('apply/step1', case=False, na=False)),
    '4. STEP 2':    pick(path.str.contains('apply/step2', case=False, na=False)),
    '5. STEP 3':    pick(path.str.contains('apply/step3', case=False, na=False)),
    '6. STEP 4':    pick(path.str.contains('apply/step4', case=False, na=False)),
}

# 3) 앞 단계 이후 120분 안에 다음 단계가 있으면 '도달'로 인정
WINDOW = 120
names = list(steps)
reached = steps[names[0]]                       # 1단계는 전부 인정
user_counts = [reached['user_uuid'].nunique()]

for name in names[1:]:
    now = steps[name]
    merged = pd.merge_asof(now, reached.rename(columns={'t': 't_prev'}).sort_values('t_prev'),
                           by='user_uuid', left_on='t', right_on='t_prev', direction='backward')
    minutes = (merged['t'] - merged['t_prev']).dt.total_seconds() / 60
    reached = merged[(minutes >= 0) & (minutes <= WINDOW)][['user_uuid', 't']].drop_duplicates()
    user_counts.append(reached['user_uuid'].nunique())

# 4) 결과 표
result = pd.DataFrame({'도달 사용자 수': user_counts}, index=names)
result['직전 단계 전환율(%)'] = (result['도달 사용자 수'] / result['도달 사용자 수'].shift(1) * 100).round(1)
result['이탈률(%)'] = (100 - result['직전 단계 전환율(%)']).round(1)
display(result)

,도달 사용자 수,직전 단계 전환율(%),이탈률(%)
1. 검색,6435,NaN,NaN
2. 상세 보기,5312,82.5,17.5
3. STEP 1,3506,66.0,34.0
4. STEP 2,3192,91.0,9.0
5. STEP 3,2624,82.2,17.8
6. STEP 4,2137,81.4,18.6


# 각 그룹 퍼널 분석 비교

In [203]:
import pandas as pd

WINDOW = 120
seg_col = 'final_segment' if 'final_segment' in final_user_scores.columns else 'user_group'

def funnel_for_group(prefix):
    # 1) 그룹 사용자의 로그만 꺼내기
    users = final_user_scores.loc[final_user_scores[seg_col].astype(str).str.startswith(prefix), 'user_uuid']
    log = log_df[log_df['user_uuid'].isin(users)][['user_uuid', 'timestamp', 'path']]
    path = log['path'].astype(str)

    # 2) 단계별 로그 모으기
    def pick(cond):
        return log.loc[cond, ['user_uuid', 'timestamp']].rename(columns={'timestamp': 't'}).sort_values('t')

    steps = {
        '1. 검색':      pick(path.str.contains('api/search', case=False, na=False)),
        '2. 상세 보기': pick(path == 'api/jobs/job_title'),
        '3. STEP 1':    pick(path.str.contains('apply/step1', case=False, na=False)),
        '4. STEP 2':    pick(path.str.contains('apply/step2', case=False, na=False)),
        '5. STEP 3':    pick(path.str.contains('apply/step3', case=False, na=False)),
        '6. STEP 4':    pick(path.str.contains('apply/step4', case=False, na=False)),
    }

    # 3) 앞 단계 이후 120분 안에 다음 단계가 있으면 도달
    names = list(steps)
    reached = steps[names[0]]
    counts = [reached['user_uuid'].nunique()]
    for name in names[1:]:
        merged = pd.merge_asof(steps[name], reached.rename(columns={'t': 't_prev'}).sort_values('t_prev'),
                               by='user_uuid', left_on='t', right_on='t_prev', direction='backward')
        minutes = (merged['t'] - merged['t_prev']).dt.total_seconds() / 60
        reached = merged[(minutes >= 0) & (minutes <= WINDOW)][['user_uuid', 't']].drop_duplicates()
        counts.append(reached['user_uuid'].nunique())
    return pd.Series(counts, index=names)

# 4) 5개 그룹 실행
groups = {'G0': 'G0', 'G1': 'G1', 'A': 'A_', 'B': 'B_', 'C': 'C_'}
counts_tbl = pd.DataFrame({g: funnel_for_group(p) for g, p in groups.items()})

# 5) 결과: 도달 사용자 수 / 이탈률
conv = counts_tbl / counts_tbl.shift(1) * 100
drop_tbl = (100 - conv).round(1)

print("=== 도달 사용자 수 ===")
display(counts_tbl)
print("\n=== 직전 단계 대비 이탈률(%) ===")
display(drop_tbl)

=== 도달 사용자 수 ===


,G0,G1,A,B,C
1. 검색,6435,2911,150,490,171
2. 상세 보기,5312,2603,143,449,161
3. STEP 1,3506,1983,134,387,150
4. STEP 2,3192,1895,132,375,150
5. STEP 3,2624,1573,129,338,143
6. STEP 4,2137,1363,125,300,137



=== 직전 단계 대비 이탈률(%) ===


,G0,G1,A,B,C
1. 검색,NaN,NaN,NaN,NaN,NaN
2. 상세 보기,17.5,10.6,4.7,8.4,5.8
3. STEP 1,34.0,23.8,6.3,13.8,6.8
4. STEP 2,9.0,4.4,1.5,3.1,0.0
5. STEP 3,17.8,17.0,2.3,9.9,4.7
6. STEP 4,18.6,13.4,3.1,11.2,4.2


In [205]:
import pandas as pd

seg_col = 'final_segment' if 'final_segment' in final_user_scores.columns else 'user_group'
g0_users = final_user_scores.loc[final_user_scores[seg_col].astype(str).str.startswith('G0'), 'user_uuid']
log = (log_df[log_df['user_uuid'].isin(g0_users)][['user_uuid', 'timestamp', 'path']]
       .sort_values(['user_uuid', 'timestamp']).copy())
p = log['path'].astype(str).str.lower()

# 자동으로 호출되는 로그는 제외 (명세서 기준)
auto = p.str.contains(r'other_jobs|companies/[^/]+/view$|widget|suggest', regex=True, na=False)
log, p = log[~auto].copy(), p[~auto]

# 세션 나누기: 같은 사용자의 활동 간격이 30분을 넘으면 새 세션
gap = log.groupby('user_uuid')['timestamp'].diff().dt.total_seconds() / 60
log['세션'] = (gap.isna() | (gap > 30)).cumsum()

flags = pd.DataFrame({
    '세션': log['세션'], 'user_uuid': log['user_uuid'], 'timestamp': log['timestamp'],
    '상세 조회':  p.eq('api/jobs/job_title'),
    '검색':       p.str.contains('api/search', na=False),
    '북마크':     p.str.contains(r'jobs/[^/]+/bookmark', regex=True, na=False),
    '기업 정보':  p.str.contains('api/companies', na=False),
    '이력서 작성': p.str.contains(r'resume|/(?:career|education|project)', regex=True, na=False),
    '지원 시작':  p.str.contains('apply/step1', na=False),
})
sess = flags.groupby('세션').agg(user_uuid=('user_uuid', 'first'), 시작=('timestamp', 'min'),
                                 상세조회=('상세 조회', 'sum'), 검색=('검색', 'sum'),
                                 북마크=('북마크', 'sum'), 기업정보=('기업 정보', 'sum'),
                                 이력서작성=('이력서 작성', 'sum'), 지원시작=('지원 시작', 'sum')).reset_index()

# 이전 세션에서 이미 지원을 시작한 적이 있는 사용자인가
sess = sess.sort_values(['user_uuid', '시작'])
started = (sess['지원시작'] > 0).astype(int)
sess['이전지원경험'] = (started.groupby(sess['user_uuid']).cumsum() - started) > 0

# 상세를 본 세션만, 같은 세션 안에서 지원 시작 여부로 나누기
v = sess[sess['상세조회'] > 0].copy()
v['결과'] = (v['지원시작'] > 0).map({True: '지원 시작', False: '지원 안 함'})

print(f"상세 조회 세션 {len(v):,}개")
print(v['결과'].value_counts())
cols = ['상세조회', '검색', '북마크', '기업정보', '이력서작성']
print("\n중앙값 비교 (세션 안 활동 횟수):")
display(v.groupby('결과')[cols].median())
print("\n해당 활동을 한 세션 비율(%):")
display((v[cols].gt(0).groupby(v['결과']).mean() * 100).round(1))
print("\n이전에 지원을 시작한 적 있는 사용자의 세션 비율(%):")
display((v.groupby('결과')['이전지원경험'].mean() * 100).round(1))

상세 조회 세션 141,914개
결과
지원 안 함    120701
지원 시작      21213
Name: count, dtype: int64

중앙값 비교 (세션 안 활동 횟수):


,상세조회,검색,북마크,기업정보,이력서작성
결과,,,,,
지원 시작,5.0,0.0,0.0,0.0,0.0
지원 안 함,2.0,0.0,0.0,0.0,0.0



해당 활동을 한 세션 비율(%):


,상세조회,검색,북마크,기업정보,이력서작성
결과,,,,,
지원 시작,100.0,44.1,20.0,37.5,39.2
지원 안 함,100.0,18.1,7.1,18.0,7.6



이전에 지원을 시작한 적 있는 사용자의 세션 비율(%):


결과
지원 시작     81.9
지원 안 함    82.9
Name: 이전지원경험, dtype: float64

### 인사이트
- 상세를 본 방문의 85%는 같은 방문에서 지원 시작으로 이어지지 않았고, 이전 지원 경험은 차이가 없었다(81.9% vs 82.9%).
- 이 표의 활동량(상세 조회, 이력서 작성 등)은 지원 시작 이후의 활동이 섞여 있어 원인으로 해석할 수 없다.

### 결론 및 다음 분석
- 결론: 이 코드만으로는 G0의 이탈 원인을 확정할 수 없다.
- 다음: 지원 시작 **이전** 활동만 세어 다시 비교하고, 같은 방식으로 G1·A·B·C와 비교해 G0만의 특징인지 확인한다.

In [206]:
# STEP 1 '이전'에 일어난 이력서 활동이 얼마나 되는지
step1_time = flags[flags['지원 시작']].groupby('세션')['timestamp'].min().rename('t_step1')
res_ev = (flags[flags['이력서 작성']].merge(step1_time, left_on='세션', right_index=True)
          .dropna(subset=['t_step1']))
print("STEP 1 이전에 발생한 이력서 활동 비율(%):", round((res_ev['timestamp'] < res_ev['t_step1']).mean() * 100, 1))

STEP 1 이전에 발생한 이력서 활동 비율(%): 39.9


In [207]:
bins = pd.cut(v['상세조회'], [0, 1, 2, 5, 10, 10**9], labels=['1회', '2회', '3~5회', '6~10회', '11회+'])
t = (v.groupby(bins, observed=True)['지원시작'].agg(lambda x: (x > 0).mean() * 100).round(1)).to_frame('지원 시작 비율(%)')
t['세션 수'] = v.groupby(bins, observed=True).size()
display(t)

,지원 시작 비율(%),세션 수
상세조회,,
1회,7.3,45477
2회,10.4,25482
3~5회,16.0,38650
6~10회,24.9,21249
11회+,34.2,11056


### 인사이트
- 이력서 작성 활동 중 STEP 1 이전에 발생한 것은 39.9%뿐이고 나머지는 이후라서, 이력서 활동은 지원 흐름의 부산물로 보고 이탈 원인에서 제외한다.
- 세션 안 상세 조회 수가 늘수록 지원 시작 비율이 올라간다(1회 7.3% → 11회+ 34.2%). 다만 지원 시작 이후의 조회까지 센 값이라 과대 추정일 수 있다.

### 결론 및 다음 분석
- 결론: 이력서 준비 부족은 근거가 없고, 조회 수와 지원의 관계는 아직 확정할 수 없다.
- 다음: 상세 조회를 STEP 1 **이전**만 세어 다시 계산하고, 같은 방식으로 G1·A·B·C와 비교한다.

In [208]:
groups = {'G0': 'G0', 'G1': 'G1', 'A': 'A_', 'B': 'B_', 'C': 'C_'}
seg_col = 'final_segment' if 'final_segment' in final_user_scores.columns else 'user_group'

def session_summary(prefix):
    users = final_user_scores.loc[final_user_scores[seg_col].astype(str).str.startswith(prefix), 'user_uuid']
    log = (log_df[log_df['user_uuid'].isin(users)][['user_uuid', 'timestamp', 'path']]
           .sort_values(['user_uuid', 'timestamp']).copy())
    p = log['path'].astype(str).str.lower()

    auto = p.str.contains(r'other_jobs|companies/[^/]+/view$|widget|suggest', regex=True, na=False)
    log, p = log[~auto].copy(), p[~auto]

    gap = log.groupby('user_uuid')['timestamp'].diff().dt.total_seconds() / 60
    log['세션'] = (gap.isna() | (gap > 30)).cumsum()
    log['상세'] = p.eq('api/jobs/job_title')
    log['step1'] = p.str.contains('apply/step1', na=False)

    # 지원 시작(STEP 1) 이전의 상세 조회만 센다
    t1 = log[log['step1']].groupby('세션')['timestamp'].min().rename('t1')
    log = log.join(t1, on='세션')
    log['상세_지원전'] = log['상세'] & (log['t1'].isna() | (log['timestamp'] < log['t1']))

    s = log.groupby('세션').agg(상세조회=('상세_지원전', 'sum'), 지원시작=('step1', 'max'))
    s = s[s['상세조회'] >= 1]
    summary = {'세션 수': len(s),
               '지원 시작 비율(%)': round(s['지원시작'].mean() * 100, 1),
               '상세 조회 중앙값': s['상세조회'].median(),
               '1~2회로 끝난 세션 비율(%)': round((s['상세조회'] <= 2).mean() * 100, 1)}
    return summary, s

results, sess_by_group = {}, {}
for g, pf in groups.items():
    results[g], sess_by_group[g] = session_summary(pf)
display(pd.DataFrame(results).T)

# G0: 지원 전 상세 조회 수별 지원 시작 비율 (다시 계산)
s0 = sess_by_group['G0']
bins = pd.cut(s0['상세조회'], [0, 1, 2, 5, 10, 10**9], labels=['1회', '2회', '3~5회', '6~10회', '11회+'])
t = s0.groupby(bins, observed=True)['지원시작'].agg(lambda x: (x > 0).mean() * 100).round(1).to_frame('지원 시작 비율(%)')
t['세션 수'] = s0.groupby(bins, observed=True).size()
display(t)

,세션 수,지원 시작 비율(%),상세 조회 중앙값,1~2회로 끝난 세션 비율(%)
G0,138090.0,12.6,2.0,51.9
G1,61345.0,19.3,2.0,50.8
A,4686.0,33.4,2.0,54.8
B,13034.0,27.1,3.0,49.3
C,6295.0,35.0,2.0,57.6


,지원 시작 비율(%),세션 수
상세조회,,
1회,7.6,45646
2회,12.2,26010
3~5회,16.0,38638
6~10회,17.0,19214
11회+,15.2,8582


### 인사이트
- 지원 전 상세 조회만 세면 조회 수가 늘어도 지원 시작 비율이 3회 이후 약 16~17%에서 더 오르지 않는다(1회 7.6% → 3~5회 16.0% → 11회+ 15.2%). 앞의 "많이 볼수록 지원한다"는 지원 이후 조회가 섞인 착시였다.
- 상세 1~2회로 끝난 세션 비율은 모든 그룹이 비슷하다(G0 51.9%, G1 50.8%, A 54.8%, B 49.3%, C 57.6%). 얕은 탐색은 G0만의 특징이 아니다.
- 방문당 지원 시작 비율은 G0가 12.6%로 가장 낮다(G1 19.3%, B 27.1%, A 33.4%, C 35.0%).

### 결론 및 다음 분석
- 결론: G0는 다른 그룹과 탐색량은 비슷한데 지원 시작으로 이어지는 비율만 낮다. 다만 G0는 정의상 지원이 적은 집단이라 분류 기준과 겹친다.
- 다음: 같은 조회 수 구간에서도 G0가 낮은지 그룹별로 비교한다.

In [209]:
rows = {}
for g, s in sess_by_group.items():
    b = pd.cut(s['상세조회'], [0, 1, 2, 5, 10**9], labels=['1회', '2회', '3~5회', '6회+'])
    rows[g] = s.groupby(b, observed=True)['지원시작'].agg(lambda x: (x > 0).mean() * 100).round(1)
display(pd.DataFrame(rows))

,G0,G1,A,B,C
상세조회,,,,,
1회,7.6,11.4,21.7,20.4,23.9
2회,12.2,20.4,39.5,31.7,42.0
3~5회,16.0,24.6,41.6,31.1,43.4
6회+,16.4,23.5,35.1,28.4,40.0


### 인사이트
- 상세 조회 수를 맞춰도 G0의 지원 시작 비율이 모든 구간에서 가장 낮다(1회 7.6%, 3~5회 16.0%, 6회+ 16.4%). G1은 11.4~24.6%, A·B·C는 20~43%였다.
- 모든 그룹에서 3회 이상 보면 비율이 더 오르지 않고 평평하다. 많이 볼수록 지원한다는 근거는 없다.

### 결론 및 다음 분석
- 결론: G0의 낮은 지원 시작 비율은 탐색량 때문이 아니다. 다만 G0는 신호 없는 일반 사용자 정의와 겹쳐 그룹 간 우열이 아니라 기준 수준으로 해석한다.
- 다음: 정의와 무관한 지원서 작성 단계(STEP 2→3, 3→4) 이탈이 어떤 상황에서 생기는지 확인한다.

In [212]:
d = next_action_v2('G0')
p = d['next_path'].fillna('')
within = (d['next_time'] - d['timestamp']).dt.total_seconds() <= 30*60

d.loc[(d['행동']=='기타') & (p=='jobs/id/id_title'), '행동'] = '다른 공고 페이지'
d.loc[(d['행동']=='기타') & (p=='jobs'), '행동'] = '목록 복귀'
d.loc[(d['행동']=='기타') & (p=='search'), '행동'] = '검색 복귀'
d.loc[(d['행동']=='기타') & p.str.contains('applications|job_offer'), '행동'] = '프로필·이력서·알림'
d.loc[d['행동']=='다른 공고 상세', '행동'] = '다른 공고 페이지'
d.loc[d['행동']=='검색 복귀', '행동'] = '검색·목록 복귀'
d.loc[d['행동']=='목록 복귀', '행동'] = '검색·목록 복귀'

print(d['행동'].value_counts(normalize=True).mul(100).round(1))
print('전체 건수:', len(d))

행동
다른 공고 페이지     63.1
세션 종료         12.7
검색·목록 복귀      10.6
기업 정보          6.5
기타             3.4
프로필·이력서·알림     2.8
공고 북마크         0.7
지원 단계          0.0
Name: proportion, dtype: float64
전체 건수: 474877


## G0 개선안: 일반 사용자(신호 없는 63%)

### 한눈에 보기
공고를 보고도 지원으로 이어지지 않는 일반 사용자가 많다. 이들은 지원하지 않은 뒤 대부분 다른 공고를 계속 둘러본다. 여러 공고를 한 화면에서 비교할 수 있게 하는 것을 제안한다.

### 무엇을 발견했나
- 공고를 본 사용자 중 34%가 이후 지원을 시작하지 않았다.
- 공고를 같은 횟수만큼 봐도 다른 그룹에 비해 G0는 지원을 시작하는 비율이 가장 낮았다.
- 공고를 3번 넘게 봐도 지원 시작 비율은 약 16~17%에서 더 오르지 않았다. (많이 보여주는 것만으로는 부족하다.)
- 지원으로 이어지지 않은 공고 조회(474,877건) 뒤의 다음 행동은 다른 공고 페이지 이동 63.1%, 검색·목록 복귀 10.6%였다. 합치면 약 74%가 바로 다른 공고를 계속 탐색했다.
- 반면 공고를 북마크로 남기는 경우는 0.7%, 바로 서비스를 나가는 경우는 12.7%였다.

### 제안
**후보 비교판**: 최근 본 공고 2~3개를 나란히 놓고, 연봉·근무지·필수 자격의 차이를 한 화면에서 비교해 준다.
- 이유: G0는 공고를 계속 넘겨 보지만 기억에 남기거나 저장하지 않는다. 비교 기준을 화면에서 대신 정리해 주면 지원 판단이 쉬워질 것으로 가정한다.

### 효과 확인 방법
비교판을 본 사용자와 보지 않은 사용자의 지원 시작 비율을 비교한다. (A/B 테스트)

### 한계
- 로그에 공고 ID가 가려져 있어, "다른 공고 페이지"가 실제로 다른 공고인지는 확정할 수 없다.
- 다음 행동 비율은 다른 그룹도 비슷하므로 G0만의 특징은 아니다. G0의 낮은 지원 시작률과 함께 나타나는 패턴으로 해석한다.
- 이 데이터로는 지원하지 않은 이유(조건이 안 맞아서인지, 구경만 했는지)를 알 수 없다. 제안은 실험으로 검증해야 한다.

# G1 개선안 도출하기 

In [215]:
counts_tbl = pd.DataFrame({
    'G1': funnel_for_group('G1'),
    'G0': funnel_for_group('G0')})
counts_tbl.index.name = '단계'

drop_tbl = (100 - counts_tbl / counts_tbl.shift(1) * 100).round(1).iloc[1:]   # 직전 단계 대비 이탈률(%)
rate_tbl = (counts_tbl / counts_tbl.iloc[0] * 100).round(1)                   # 첫 단계 대비 도달률(%)

print('① 도달 인원(명)');            display(counts_tbl)
print('② 직전 단계 대비 이탈률(%)');  display(drop_tbl)
print('③ 첫 단계 대비 도달률(%)');    display(rate_tbl)

① 도달 인원(명)


,G1,G0
단계,,
1. 검색,2911,6435
2. 상세 보기,2603,5312
3. STEP 1,1983,3506
4. STEP 2,1895,3192
5. STEP 3,1573,2624
6. STEP 4,1363,2137


② 직전 단계 대비 이탈률(%)


,G1,G0
단계,,
2. 상세 보기,10.6,17.5
3. STEP 1,23.8,34.0
4. STEP 2,4.4,9.0
5. STEP 3,17.0,17.8
6. STEP 4,13.4,18.6


③ 첫 단계 대비 도달률(%)


,G1,G0
단계,,
1. 검색,100.0,100.0
2. 상세 보기,89.4,82.5
3. STEP 1,68.1,54.5
4. STEP 2,65.1,49.6
5. STEP 3,54.0,40.8
6. STEP 4,46.8,33.2


In [216]:
print(final_user_scores.columns.tolist())
print(final_user_scores[final_user_scores[seg_col].astype(str).str.startswith('G1')].head(3).T)

['user_uuid', 'volume_score', 'concentration_score', 'diversity_score', 'view_score', 'bookmark_score', 'volume_null', 'concentration_null', 'diversity_null', 'view_null', 'bookmark_null', 'total_score', 'analysis_group', 'final_segment']
                                                        2  \
user_uuid            40196d59-ab7c-4951-8a5b-eb87aac0a08b   
volume_score                                            0   
concentration_score                                     0   
diversity_score                                         1   
view_score                                              0   
bookmark_score                                          0   
volume_null                                             0   
concentration_null                                      0   
diversity_null                                          0   
view_null                                               0   
bookmark_null                                           0   
total_score                  

In [217]:
signal_cols = {
    '대량성':        'volume_score',
    '집중성':        'concentration_score',
    '다양성':        'diversity_score',
    '공고 탐색성':   'view_score',
    '북마크 탐색성': 'bookmark_score',
}

g1 = final_user_scores[final_user_scores[seg_col].astype(str).str.startswith('G1')]

# 신호별 인원 확인
sig_n = pd.Series({name: int((g1[col] == 1).sum()) for name, col in signal_cols.items()})
print('신호별 인원(명)'); display(sig_n.to_frame('인원').assign(비율=(sig_n / len(g1) * 100).round(1)))

def funnel_for_users(users, window=120):
    log = log_df[log_df['user_uuid'].isin(users)][['user_uuid','timestamp','path']]
    path = log['path'].astype(str)
    def pick(cond):
        return log.loc[cond, ['user_uuid','timestamp']].rename(columns={'timestamp':'t'}).sort_values('t')
    steps = {
        '1. 검색':    pick(path.str.contains('api/search', case=False, na=False)),
        '2. 상세 보기': pick(path == 'api/jobs/job_title'),
        '3. STEP 1':  pick(path.str.contains('apply/step1', case=False, na=False)),
        '4. STEP 2':  pick(path.str.contains('apply/step2', case=False, na=False)),
        '5. STEP 3':  pick(path.str.contains('apply/step3', case=False, na=False)),
        '6. STEP 4':  pick(path.str.contains('apply/step4', case=False, na=False)),
    }
    names = list(steps)
    reached = steps[names[0]]
    counts = [reached['user_uuid'].nunique()]
    for name in names[1:]:
        merged = pd.merge_asof(
            steps[name], reached.rename(columns={'t':'t_prev'}).sort_values('t_prev'),
            by='user_uuid', left_on='t', right_on='t_prev', direction='backward')
        minutes = (merged['t'] - merged['t_prev']).dt.total_seconds() / 60
        reached = merged[(minutes >= 0) & (minutes <= window)][['user_uuid','t']].drop_duplicates()
        counts.append(reached['user_uuid'].nunique())
    return pd.Series(counts, index=names)

sig_counts = pd.DataFrame({
    name: funnel_for_users(g1.loc[g1[col] == 1, 'user_uuid'])
    for name, col in signal_cols.items()})
sig_counts.index.name = '단계'

sig_drop = (100 - sig_counts / sig_counts.shift(1) * 100).round(1).iloc[1:]
sig_rate = (sig_counts / sig_counts.iloc[0] * 100).round(1)

print('① 도달 인원(명)');            display(sig_counts)
print('② 직전 단계 대비 이탈률(%)');  display(sig_drop)
print('③ 첫 단계 대비 도달률(%)');    display(sig_rate)

신호별 인원(명)


,인원,비율
대량성,134,4.6
집중성,75,2.6
다양성,561,19.1
공고 탐색성,2099,71.4
북마크 탐색성,72,2.4


① 도달 인원(명)


,대량성,집중성,다양성,공고 탐색성,북마크 탐색성
단계,,,,,
1. 검색,134,75,556,2074,72
2. 상세 보기,132,63,516,1826,66
3. STEP 1,122,52,418,1343,48
4. STEP 2,119,51,406,1274,45
5. STEP 3,115,49,341,1029,39
6. STEP 4,112,47,303,871,30


② 직전 단계 대비 이탈률(%)


,대량성,집중성,다양성,공고 탐색성,북마크 탐색성
단계,,,,,
2. 상세 보기,1.5,16.0,7.2,12.0,8.3
3. STEP 1,7.6,17.5,19.0,26.5,27.3
4. STEP 2,2.5,1.9,2.9,5.1,6.2
5. STEP 3,3.4,3.9,16.0,19.2,13.3
6. STEP 4,2.6,4.1,11.1,15.4,23.1


③ 첫 단계 대비 도달률(%)


,대량성,집중성,다양성,공고 탐색성,북마크 탐색성
단계,,,,,
1. 검색,100.0,100.0,100.0,100.0,100.0
2. 상세 보기,98.5,84.0,92.8,88.0,91.7
3. STEP 1,91.0,69.3,75.2,64.8,66.7
4. STEP 2,88.8,68.0,73.0,61.4,62.5
5. STEP 3,85.8,65.3,61.3,49.6,54.2
6. STEP 4,83.6,62.7,54.5,42.0,41.7


## G1 신호별 퍼널 분석

### 인사이트
- G1의 90%는 공고 탐색성(71.4%)과 다양성(19.1%)이다. 나머지 세 신호는 인원이 적어(72~134명) 참고용이다.
- 두 신호 모두 작성 중간(STEP 2→3)에서 가장 많이 이어지지 않았다. (공고 탐색성 19.2%, 다양성 16.0%)
- 대량성·집중성의 낮은 이탈, 공고 탐색성의 상세→STEP 1 이탈은 정의와 겹쳐서 해석하지 않는다.

### 결론
G1은 인원이 가장 많은 **공고 탐색성**을 중심으로, 작성 중간(STEP 2→3) 이탈을 개선 대상으로 한다.

### 다음 분석
STEP 2 요청이 한 사람당 약 20번 찍혀서 건 단위는 못 쓴다. **사용자 단위**로 STEP 2 이후 STEP 3에 영영 가지 않은 비율과 나중에 돌아온 비율을 비교한다.

In [218]:
def stall_analysis_users(users, from_step='apply/step2', to_step='apply/step3', window=120, gap_end=30):
    log = log_df[log_df['user_uuid'].isin(users)][['user_uuid','timestamp','path']].copy()
    log['path'] = log['path'].astype(str)
    log = log[~log['path'].str.contains(AUTO, case=False, regex=True)]
    log = log.sort_values(['user_uuid','timestamp']).reset_index(drop=True)

    log['to_t'] = log['timestamp'].where(log['path'].str.contains(to_step, case=False))
    g = log.groupby('user_uuid')
    log['next_to'] = g['to_t'].bfill()
    log['next_path'] = g['path'].shift(-1)
    log['next_time'] = g['timestamp'].shift(-1)

    f = log[log['path'].str.contains(from_step, case=False)].copy()
    gap_to = (f['next_to'] - f['timestamp']).dt.total_seconds() / 60
    f['결과'] = np.select([gap_to <= window, gap_to > window],
                          ['바로 완료(2시간 내)', '나중에 재개'], default='재개 안 함')

    s = f[f['결과'] != '바로 완료(2시간 내)'].copy()
    gap = (s['next_time'] - s['timestamp']).dt.total_seconds() / 60
    p = s['next_path'].fillna('')
    s['직후 행동'] = np.select(
        [gap.isna() | (gap > gap_end),
         p.str.contains('apply/step', case=False),
         p.str.contains('jobs/id/id_title|api/jobs/job_title', case=False),
         p.str.contains('search|^jobs$', case=False),
         p.str.contains('resume|profile|applications', case=False)],
        ['세션 종료','지원 단계 내 이동','다른 공고 페이지','검색·목록 복귀','프로필·이력서'],
        default='기타')
    return (f['결과'].value_counts(normalize=True).mul(100).round(1),
            s['직후 행동'].value_counts(normalize=True).mul(100).round(1), len(f))

targets = {
    'G1-공고 탐색성': g1.loc[g1['view_score']==1, 'user_uuid'],
    'G1-다양성':      g1.loc[g1['diversity_score']==1, 'user_uuid'],
}
for name, users in targets.items():
    r1, r2, n = stall_analysis_users(users)
    print(f'\n=== {name} (STEP 2 도달 {n}건) ===')
    print(r1); print('\n멈춘 직후 행동:'); print(r2)


=== G1-공고 탐색성 (STEP 2 도달 42637건) ===
결과
바로 완료(2시간 내)    75.8
나중에 재개          16.2
재개 안 함           8.0
Name: proportion, dtype: float64

멈춘 직후 행동:
직후 행동
지원 단계 내 이동    54.1
기타            44.7
세션 종료          0.6
다른 공고 페이지      0.3
검색·목록 복귀       0.2
프로필·이력서        0.1
Name: proportion, dtype: float64

=== G1-다양성 (STEP 2 도달 22055건) ===
결과
바로 완료(2시간 내)    73.4
나중에 재개          20.5
재개 안 함           6.0
Name: proportion, dtype: float64

멈춘 직후 행동:
직후 행동
지원 단계 내 이동    58.4
기타            39.9
세션 종료          0.8
다른 공고 페이지      0.4
검색·목록 복귀       0.3
프로필·이력서        0.2
Name: proportion, dtype: float64


## G1 작성 중단(STEP 2→3) 건 단위 분석

### 인사이트
- STEP 2 기록이 42,637건(공고 탐색성 사용자 2,099명)이라 한 사람당 약 20번씩 찍힌다. 같은 요청이 반복 기록되는 것으로 보인다.
- 2시간 안에 STEP 3로 이어진 건이 75.8%, 나중에 이어진 건이 16.2%, 이어지지 않은 건이 8.0%다. (다양성: 73.4% / 20.5% / 6.0%)
- 멈춘 직후 행동은 "지원 단계 내 이동" 54.1%이고, 세션 종료 0.6%, 다른 공고 이동 0.3%로 거의 없다. 즉 대부분은 떠난 게 아니라 같은 요청이 반복된 것이다.

### 결론
건 단위로는 "작성 중 멈춤"과 "반복 요청"을 구분할 수 없어서 이어쓰기냐 사전 안내냐를 정할 수 없다. 이 결과는 근거로 쓰지 않는다.

### 다음 분석
**사용자 단위**로 다시 본다. 첫 STEP 2 이후 STEP 3에 가지 않은 비율, 2시간 뒤 도달한 비율을 비교한다.

In [219]:
# 1) STEP 2 경로 종류와 사용자당 요청 횟수 확인
v = g1.loc[g1['view_score']==1, 'user_uuid']
s2 = log_df[log_df['user_uuid'].isin(v) & log_df['path'].astype(str).str.contains('apply/step2', case=False)]
print(s2['path'].value_counts().head(10))
print('사용자당 STEP 2 요청 수:'); print(s2.groupby('user_uuid').size().describe().round(1))

path
jobs/id/apply/step2        22752
api/jobs/id/apply/step2    19885
Name: count, dtype: int64
사용자당 STEP 2 요청 수:
count    2096.0
mean       20.3
std        23.7
min         1.0
25%         6.0
50%        13.0
75%        26.0
max       233.0
dtype: float64


In [220]:
# 2) 사용자 단위 재개율 (사용자의 첫 STEP 2 기준)
def user_level_stall(users, from_step='apply/step2', to_step='apply/step3', window=120):
    log = log_df[log_df['user_uuid'].isin(users)][['user_uuid','timestamp','path']].copy()
    log['path'] = log['path'].astype(str)
    first_from = (log[log['path'].str.contains(from_step, case=False)]
                  .groupby('user_uuid')['timestamp'].min().rename('t_from'))
    to_t = log[log['path'].str.contains(to_step, case=False)][['user_uuid','timestamp']]
    m = to_t.merge(first_from.reset_index(), on='user_uuid')
    m['gap'] = (m['timestamp'] - m['t_from']).dt.total_seconds() / 60
    m = m[m['gap'] >= 0]
    first_to = m.groupby('user_uuid')['gap'].min()           # 첫 STEP 2 이후 가장 빠른 STEP 3까지 시간(분)

    res = first_from.to_frame().join(first_to.rename('gap'))
    res['결과'] = np.select(
        [res['gap'] <= window, res['gap'] > window],
        ['2시간 내 STEP 3 도달', '2시간 이후 도달'], default='STEP 3 도달 안 함')
    return res['결과'].value_counts().to_frame('인원').assign(
        비율=lambda x: (x['인원'] / x['인원'].sum() * 100).round(1))

for name, users in targets.items():
    print(f'\n=== {name} ===')
    display(user_level_stall(users))


=== G1-공고 탐색성 ===


,인원,비율
결과,,
2시간 내 STEP 3 도달,1496,71.4
2시간 이후 도달,364,17.4
STEP 3 도달 안 함,236,11.3



=== G1-다양성 ===


,인원,비율
결과,,
2시간 내 STEP 3 도달,406,72.5
2시간 이후 도달,136,24.3
STEP 3 도달 안 함,18,3.2


## G1 작성 중단(STEP 2→3) 사용자 단위 분석

### 인사이트
- STEP 2가 페이지·API 두 경로로 찍혀 한 사람당 중앙값 13건 기록된다. 건 단위 분석은 쓸 수 없다.
- 사용자 단위로 보면 공고 탐색성의 71.4%가 2시간 내 STEP 3에 도달했고, 11.3%는 끝내 도달하지 않았다. 다양성은 도달하지 않은 비율이 3.2%뿐이다.
- 퍼널의 19.2%(2→3 이탈) 중 상당수는 나중에 STEP 3에 도달하는 사용자다.

### 결론
작성 중간이 개선 대상이라는 근거는 약하다. 특히 다양성은 해당하지 않고, 공고 탐색성도 영영 이어지지 않는 비율은 11.3%에 그친다. "2시간 이후 도달"이 이어쓰기인지 다른 지원인지는 구분되지 않는다.

### 다음 분석
2시간 이후 도달한 사용자 중 STEP 2와 STEP 3 사이에 STEP 1이 다시 찍힌 비율을 본다. 있으면 새 지원서, 없으면 같은 지원서 이어쓰기로 해석한다.

In [221]:
def resume_check(users, window=120):
    log = log_df[log_df['user_uuid'].isin(users)][['user_uuid','timestamp','path']].copy()
    log['path'] = log['path'].astype(str)
    first2 = (log[log['path'].str.contains('apply/step2', case=False)]
              .groupby('user_uuid')['timestamp'].min().rename('t2'))
    s3 = log[log['path'].str.contains('apply/step3', case=False)].merge(first2.reset_index(), on='user_uuid')
    s3 = s3[s3['timestamp'] >= s3['t2']]
    t3 = s3.groupby('user_uuid')['timestamp'].min().rename('t3')
    late = pd.concat([first2, t3], axis=1).dropna()
    late = late[(late['t3'] - late['t2']).dt.total_seconds() / 60 > window]   # 2시간 이후 도달

    s1 = log[log['path'].str.contains('apply/step1', case=False)][['user_uuid','timestamp']]
    m = s1.merge(late.reset_index(), on='user_uuid')
    new_app = m[(m['timestamp'] > m['t2']) & (m['timestamp'] < m['t3'])]['user_uuid'].nunique()
    return len(late), new_app, round(new_app / len(late) * 100, 1)

for name, users in targets.items():
    n, k, pct = resume_check(users)
    print(f'{name}: 2시간 이후 도달 {n}명 중 중간에 STEP 1 재시작 {k}명 ({pct}%)')

G1-공고 탐색성: 2시간 이후 도달 364명 중 중간에 STEP 1 재시작 359명 (98.6%)
G1-다양성: 2시간 이후 도달 136명 중 중간에 STEP 1 재시작 131명 (96.3%)


## G1 작성 중단 이후 행동 분석

### 인사이트
- 첫 STEP 2 이후 2시간 안에 STEP 3까지 간 사용자는 공고 탐색성 71.4%, 다양성 72.5%다. 나머지 약 28%는 첫 지원서를 STEP 2에서 끝내지 못했다.
- 이 중 나중에 STEP 3에 도달한 사용자의 96~99%는 사이에 STEP 1을 다시 시작했다. 멈춘 지원서를 이어서 쓴 게 아니라 처음부터 다시 시작한 것으로 보인다.

### 결론
G1은 작성 중간에 멈추면, 돌아와도 이어쓰지 않고 처음부터 다시 쓴다. 개선 대상은 "작성 전 부담 안내"보다 "멈춘 지점으로 돌아오는 경로"가 데이터에 더 가깝다.

### 한계
- 공고 ID가 가려져 있어서, 다시 시작한 지원서가 같은 공고인지 다른 공고인지는 알 수 없다.
- STEP 1 재기록은 같은 지원서에서 앞 단계로 돌아간 경우도 포함할 수 있다.
- 첫 STEP 2만 기준으로 봤으므로 여러 번 지원하는 사용자의 이후 지원서는 반영하지 않았다.

## G1 개선안: 단일 신호 사용자(28.6%)

### 한눈에 보기
G1의 90%는 공고를 보고 빠르게 지원을 시작하는 사용자(공고 탐색성 71.4%)와 여러 직무에 지원하는 사용자(다양성 19.1%)다. 이들은 지원서를 쓰다 멈추면 돌아와도 이어쓰지 않고 처음부터 다시 시작한다. 멈춘 단계로 바로 돌아오게 하는 것을 제안한다.

### 무엇을 발견했나
- 두 신호 모두 지원서 작성 중간(STEP 2→3)에서 이어지지 않은 비율이 가장 컸다. (공고 탐색성 19.2%, 다양성 16.0%)
- 첫 STEP 2 이후 2시간 안에 STEP 3까지 간 사용자는 71~73%였고, 나머지 약 28%는 첫 지원서를 STEP 2에서 끝내지 못했다.
- 나중에 STEP 3에 도달한 사용자의 96~99%는 그 사이에 STEP 1을 다시 시작했다. 이어서 쓴 게 아니라 처음부터 다시 쓴 것으로 보인다.

### 제안
**멈춘 단계로 바로 복귀**: 지원서를 쓰다 멈춘 사용자가 다시 들어오면, 처음부터가 아니라 마지막으로 멈춘 단계(STEP 2)로 바로 이동시킨다.
- 이유: 멈춘 뒤 돌아와도 STEP 1부터 다시 시작하는 사용자가 대부분이므로, 재진입 경로를 줄이면 작성 완료가 늘어날 것으로 가정한다.

### 효과 확인 방법
멈춘 단계로 복귀한 사용자와 그렇지 않은 사용자의 지원 완료 비율(STEP 4 도달), 재작성(STEP 1 재시작) 비율을 비교한다. (A/B 테스트)

### 한계
- 공고 ID가 가려져 있어, 다시 시작한 지원서가 같은 공고인지 다른 공고인지는 확정할 수 없다.
- STEP 1 재기록은 같은 지원서에서 앞 단계로 돌아간 경우도 포함할 수 있다.
- 대량성·집중성·북마크 탐색성은 인원이 적어(72~134명) 이번 제안의 근거에서 제외했다.
- 제안의 효과는 이 분석으로 알 수 없고, 실험으로 검증해야 한다.

# A개선안 도출하기

In [ ]:
# 1) 지원 데이터 + 공고 정보, 그룹 라벨
def short_seg(s):
    s = str(s)
    return 'G0' if s.startswith('G0') else 'G1' if s.startswith('G1') else s[0]   # A_, B_, C_ → A, B, C

seg = final_user_scores[['user_uuid', seg_col]].assign(group=lambda x: x[seg_col].map(short_seg))[['user_uuid','group']]

ap = (app_df[['user_uuid','application_uuid','company_uuid','job_uuid','cdate_kst']]
      .merge(job_df.drop_duplicates('job_uuid')[['job_uuid','job_field','career_type_string','end_date']],
             on='job_uuid', how='left')
      .merge(seg, on='user_uuid', how='inner'))

ap['t'] = pd.to_datetime(ap['cdate_kst'], errors='coerce', utc=True).dt.tz_convert('Asia/Seoul')
ap['day'] = ap['t'].dt.date
ap['end'] = pd.to_datetime(ap['end_date'], errors='coerce', utc=True).dt.tz_convert('Asia/Seoul')
ap['days_left'] = (ap['end'] - ap['t']).dt.total_seconds() / 86400
ap['near_deadline'] = ap['days_left'].between(0, 3)          # 마감 3일 이내 지원

# 2) 사용자별 지표
u = ap.groupby('user_uuid').agg(
    group=('group','first'),
    지원수=('application_uuid','nunique'),
    활동일수=('day','nunique'),
    회사수=('company_uuid','nunique'),
    직무분야수=('job_field','nunique'),
    마감임박비율=('near_deadline','mean'),
    마감일_결측=('end','count'))
u['활동일당_지원'] = u['지원수'] / u['활동일수']
u['같은회사_중복비율'] = 1 - u['회사수'] / u['지원수']          # 이미 지원한 회사에 또 지원한 비율

# 3) 그룹별 요약 (중앙값)
cols = ['지원수','활동일당_지원','직무분야수','같은회사_중복비율','마감임박비율']
summary = u.groupby('group')[cols].median().round(2).T
summary.loc['인원'] = u.groupby('group').size()
display(summary[['G0','G1','A','B','C']])

# 4) 경력 유형 분포 (지원한 공고 기준, %)
ct = pd.crosstab(ap['career_type_string'], ap['group'], normalize='columns').mul(100).round(1)
display(ct[['G0','G1','A','B','C']])

group,G0,G1,A,B,C
지원수,2.0,4.00,38.00,10.0,37.00
활동일당_지원,1.0,1.53,5.67,2.5,6.00
직무분야수,1.0,1.00,2.00,2.0,2.00
같은회사_중복비율,0.0,0.00,0.05,0.0,0.04
마감임박비율,0.0,0.00,0.04,0.0,0.03
인원,6484.0,2941.00,151.00,492.0,171.00


group,G0,G1,A,B,C
career_type_string,,,,,
경력,37.3,38.7,42.4,39.7,47.6
"경력,인턴",0.1,0.1,0.1,0.0,0.1
신입,4.5,3.8,3.1,3.6,2.6
"신입,경력",43.9,42.7,37.6,43.6,36.3
"신입,경력,인턴",10.2,10.2,12.2,9.2,9.9
"신입,인턴",1.8,2.2,2.3,1.8,1.7
인턴,2.3,2.4,2.3,2.1,1.8


## A 지원 패턴 분석

### 인사이트
- A는 지원 건수 중앙값 38건, 활동일당 5.7건으로 매우 많이 몰아서 지원한다. 다만 이건 A의 정의(대량성·집중성)와 겹치는 숫자다.
- 정의 밖의 지표에서는 A의 특징이 없었다. 같은 회사 중복 지원은 5%, 마감 3일 이내 지원은 4%로 낮고, 경력 유형 분포도 다른 그룹과 비슷하다.
- A와 C는 지원 규모(38건 / 37건, 활동일당 5.7건 / 6.0건)가 거의 같다. 둘의 차이는 공고를 보고 바로(3분 이내) 지원하느냐뿐이다.

### 결론
기업·마감·경력 측면에서는 A만의 개선 근거를 찾지 못했다. A는 "많이 지원하는 사용자"라는 특징만 확인됐고, 반복 지원의 부담은 아직 확인하지 않았다.

### 다음 분석
많이 지원하는 사용자의 **지원서 작성 시간**이 지원 횟수가 늘수록 줄어드는지 본다. 줄지 않으면 매번 같은 내용을 다시 입력하는 반복 부담이 있다는 근거가 된다.

In [229]:
# 0) user_uuid 자료형 통일
def fix(df):
    df = df.copy()
    df['user_uuid'] = df['user_uuid'].astype(str)
    return df

# 1) 그룹 라벨
def short_seg(s):
    s = str(s)
    return 'G0' if s.startswith('G0') else 'G1' if s.startswith('G1') else s[0]   # A_, B_, C_ → A, B, C

seg = (fix(final_user_scores)[['user_uuid', seg_col]]
       .assign(group=lambda x: x[seg_col].map(short_seg))[['user_uuid','group']])

# 2) 지원 완료 시각 ← 직전 STEP 1 시작 ← 직전 상세 조회
lg = fix(log_df)[['user_uuid','timestamp','path']]
lg['path'] = lg['path'].astype(str)

s1 = (lg[lg['path'].str.contains('apply/step1', case=False, na=False)]
      [['user_uuid','timestamp']].rename(columns={'timestamp':'t_start'}).sort_values('t_start'))
det = (lg[lg['path'] == 'api/jobs/job_title']
       [['user_uuid','timestamp']].rename(columns={'timestamp':'t_det'}).sort_values('t_det'))

a = fix(app_df)[['user_uuid','application_uuid','cdate']].copy()
a['t_apply'] = pd.to_datetime(a['cdate'], errors='coerce', utc=True)
a = a.dropna(subset=['t_apply']).merge(seg, on='user_uuid', how='inner').sort_values('t_apply')

ap_t = pd.merge_asof(a, s1, by='user_uuid', left_on='t_apply', right_on='t_start',
                     direction='backward', tolerance=pd.Timedelta('120min'))
ap_t = ap_t.dropna(subset=['t_start']).sort_values('t_start')
ap_t = pd.merge_asof(ap_t, det, by='user_uuid', left_on='t_start', right_on='t_det',
                     direction='backward', tolerance=pd.Timedelta('120min'))

col = '조회→지원시작(분)'
ap_t[col] = (ap_t['t_start'] - ap_t['t_det']).dt.total_seconds() / 60
print('STEP 1 연결 성공:', len(ap_t), '/', len(a), f'({len(ap_t)/len(a)*100:.1f}%)')

# 3) 그룹별 분포
sub = ap_t.dropna(subset=[col])
q = sub.groupby('group')[col].quantile([.25,.5,.75,.9]).unstack().round(1).loc[['G0','G1','A','B','C']]
q['3분 이하 비율(%)'] = sub.groupby('group')[col].apply(lambda s: (s <= 3).mean()*100).round(1)
q['건수'] = sub.groupby('group')[col].count()
display(q)

STEP 1 연결 성공: 83619 / 92526 (90.4%)


,0.25,0.5,0.75,0.9,3분 이하 비율(%),건수
group,,,,,,
G0,1.8,4.7,14.0,36.0,37.6,18277
G1,1.3,3.1,9.4,29.3,49.4,17587
A,1.8,4.7,14.2,35.8,38.2,6611
B,1.0,2.2,5.9,17.3,59.6,7706
C,1.0,2.4,7.0,20.2,55.9,10423


## A 조회→지원 시작 시간 분석

### 인사이트
- 조회부터 지원 시작까지 걸리는 시간이 A는 G0와 거의 같다. (중앙값 A 4.7분 / G0 4.7분, 75% 지점 14.2분 / 14.0분, 90% 지점 35.8분 / 36.0분)
- 반대로 B(2.2분), C(2.4분), G1(3.1분)은 더 빠르다.
- 지원 규모는 A가 G0의 약 19배(중앙값 38건 / 2건)인데, 지원 속도는 G0와 같다.
- 즉 A는 "한 공고를 오래 보는 신중한 사용자"가 아니라, **평소 속도로 지원을 아주 많이 하는 사용자**다. 앞서 세운 "한 공고를 오래 본다"는 가설은 맞지 않았다.

### 결론
A의 특징은 판단 속도가 아니라 **지원 횟수 자체**다. 개선 근거는 많이 지원하는 과정(반복)에서 찾아야 한다.

### 다음 분석
지원 횟수가 늘수록 지원서 작성 시간이 줄어드는지 본다. 줄지 않으면 같은 내용을 매번 다시 쓰는 반복 부담이 있다는 근거가 된다.

### 한계
- 로그와 지원 기록의 연결 성공률은 90.4%다.
- 조회→지원 시작 시간은 화면에 머문 시간이라서 실제로 읽은 시간과 다를 수 있다.

In [230]:
ap_t['작성분'] = (ap_t['t_apply'] - ap_t['t_start']).dt.total_seconds() / 60
ap_t = ap_t[(ap_t['작성분'] >= 0) & (ap_t['작성분'] <= 120)].copy()

ap_t = ap_t.sort_values('t_apply')
ap_t['순번'] = ap_t.groupby('user_uuid').cumcount() + 1
ap_t['구간'] = pd.cut(ap_t['순번'], [0, 1, 5, 20, 10**6],
                      labels=['1번째', '2~5번째', '6~20번째', '21번째+'])

sub = ap_t[ap_t['group'].isin(['A', 'B', 'C'])]
t = sub.groupby(['구간', 'group'], observed=True)['작성분'].median().unstack().round(1)
n = sub.groupby(['구간', 'group'], observed=True).size().unstack()

print('지원서 작성 소요시간 중앙값(분)'); display(t)
print('건수'); display(n)

지원서 작성 소요시간 중앙값(분)


group,A,B,C
구간,,,
1번째,1.9,1.1,1.2
2~5번째,0.9,0.5,0.5
6~20번째,0.6,0.4,0.3
21번째+,0.5,0.4,0.3


건수


group,A,B,C
구간,,,
1번째,150,492,171
2~5번째,600,1832,684
6~20번째,2099,3323,2298
21번째+,5459,3816,8958


## A 작성 소요시간 분석

### 인사이트
- A의 지원서 작성 시간은 지원 횟수가 늘수록 줄었다. (중앙값 1번째 1.9분 → 2~5번째 0.9분 → 6~20번째 0.6분 → 21번째 이후 0.5분)
- 6번째 지원부터는 작성에 30~40초밖에 안 걸린다. B, C도 비슷하게 줄었다.
- A는 지원 규모가 큰 만큼 21번째 이후 지원이 5,459건으로 가장 많다.

### 결론
많이 지원하는 사용자에게 지원서 작성의 반복 부담은 없다. 지원이 매우 빠르게 반복되는 구조여서, "작성 재사용" 방향은 근거가 없어 접는다.

### 다음 분석
지원이 많고 쉬운 만큼 남는 문제는 **지원한 뒤의 관리**일 수 있다. 지원 후에 내 지원 내역 페이지를 얼마나 자주 방문하는지, A가 다른 그룹보다 확인 수요가 큰지 본다.

### 한계
- 작성 시간은 STEP 1 이후 멈춘 시간까지 포함할 수 있어 중앙값으로 본다.
- 지원 순번은 로그와 연결된 지원(90.4%)만 센 것이다.

In [ ]:
lg = fix(log_df)[['user_uuid','timestamp','path']]
lg['path'] = lg['path'].astype(str)

visit = lg[lg['path'].str.contains('^@user_id/applications', case=False, na=False)]
offer = lg[lg['path'].str.contains('job_offer/received', case=False, na=False)]

n_app = fix(app_df).groupby('user_uuid')['application_uuid'].nunique().rename('지원수')

u = seg.set_index('user_uuid').join(n_app).fillna({'지원수': 0})
u['지원내역_방문수'] = visit.groupby('user_uuid').size().reindex(u.index).fillna(0)
u['받은제안_방문수'] = offer.groupby('user_uuid').size().reindex(u.index).fillna(0)

g = u.groupby('group')
res = pd.DataFrame({
    '인원': g.size(),
    '지원내역 방문자 비율(%)': g['지원내역_방문수'].apply(lambda s: (s > 0).mean()*100).round(1),
    '방문자 중 방문수 중앙값': g['지원내역_방문수'].apply(lambda s: s[s > 0].median()).round(1),
    '지원 10건당 방문수': (g['지원내역_방문수'].sum() / g['지원수'].sum() * 10).round(2),
    '받은제안 방문자 비율(%)': g['받은제안_방문수'].apply(lambda s: (s > 0).mean()*100).round(1),
}).loc[['G0','G1','A','B','C']]
display(res)

① 지원 1건당 상세 조회 수 (중앙값)


group,A,B,C
하루 지원 수,,,
1건,4.00,5.00,5.00
2~4건,2.00,3.00,3.00
5~9건,1.13,1.76,1.56
10건+,0.67,1.10,0.80


해당 일수(사용자-일)


group,A,B,C
하루 지원 수,,,
1건,442,1129,473
2~4건,486,1323,655
5~9건,268,368,399
10건+,225,118,335


② 지원 직후 흐름


,지원건수,30분 내 이어짐(%),이어진 경우 간격 중앙값(분)
group,,,
G0,26941,37.7,1.9
G1,24010,43.1,1.8
A,8308,47.2,2.7
B,9463,55.7,1.5
C,12111,67.0,1.6


## A 대량 지원 방식 분석

### 인사이트
- 하루 지원이 늘수록 지원 1건당 상세 조회 수가 줄었다. A는 1건 4.0 → 2~4건 2.0 → 5~9건 1.13 → 10건 이상 0.67이다.
- A는 모든 구간에서 B, C보다 조회가 적다. (10건 이상: A 0.67 / B 1.10 / C 0.80) 하루 10건 이상 지원하는 날이 A 사용자-일의 약 16%다.
- 지원 직후 30분 안에 다른 공고 상세로 이어지는 비율은 A 47.2%, C 67.0%, B 55.7%, G1 43.1%, G0 37.7%다. A는 같은 대량 지원 그룹인 C보다 20%p 낮고, 이어질 때 간격도 더 길다(2.7분 / 1.6분).

### 결론
A는 하루에 많이 몰아서 지원할수록 공고당 정보 확인(상세 조회)이 줄고, 지원 후 다음 공고로 이어지는 흐름은 C보다 약하다. 다만 차이가 크지 않고 근거는 가설 수준이다.

### 한계
- 지원당 조회가 1 미만인 건 그날 상세 조회 없이 지원한 건이 있다는 뜻이다. 전날 조회, 북마크, 로그에 안 잡히는 경로일 수 있다.
- 조회가 적다고 지원의 질이 낮다는 근거는 없다. 합격 등 결과 데이터가 없다.
- C의 높은 이어짐은 "공고 탐색성(조회 후 빠른 지원)" 정의와 일부 겹친다.

## A 개선안: 다수 공고 지원형(151명)

### 한눈에 보기
A는 지원 행동 신호(대량성·집중성·다양성)가 복합적으로 나타나지만, 조회 후 바로 지원하는 공고 탐색성은 없는 그룹이다. 하루에 지원을 몰아서 할수록 공고 1건당 확인(상세 조회)이 줄어든다. 여러 공고를 한 화면에서 확인하고 지원하는 방식을 제안한다.

### 무엇을 발견했나
- A는 지원 건수 중앙값이 38건, 활동일당 5.7건이다. (정의와 겹치는 지표라서 특징 설명에만 쓴다.)
- 작성 시간은 지원 횟수가 늘수록 줄었다. (1번째 1.9분 → 21번째 이후 0.5분) 반복 입력 부담은 보이지 않았다.
- 하루 지원이 많을수록 지원 1건당 상세 조회가 줄었다. (1건 4.0 → 5~9건 1.13 → 10건 이상 0.67) 모든 구간에서 B, C보다 조회가 적었다.
- 기업·마감·경력 측면에서는 다른 그룹과 뚜렷한 차이가 없었다.

### 제안
**일괄 확인 후 지원**: 관심 공고 여러 개를 한 화면에 모아 마감일·자격·근무조건을 한 번에 확인하고 지원하게 한다.
- 이유: 하루에 많이 지원할수록 공고별로 상세 페이지를 열어 보는 횟수가 줄므로, 한 화면에서 확인할 수 있게 하면 대량 지원 흐름에 맞을 것으로 가정한다.

### 효과 확인 방법
일괄 확인 화면을 쓴 사용자와 쓰지 않은 사용자의 지원 완료율과 지원 후 취소·철회 비율을 비교한다. (A/B 테스트)

### 한계
- 지원당 조회가 1 미만인 경우는 전날 조회·북마크·로그에 안 잡히는 경로일 수 있다.
- 조회가 적다는 것이 지원의 질이 낮다는 뜻은 아니다. 합격 등 결과 데이터가 없다.
- 인원이 151명으로 적고, 분석은 관찰에서 나온 가설이다. 실험으로 검증이 필요하다.

# B 개선안 도출

In [233]:
# ── ① 지원 직전 기업 정보 확인 비율 ──
comp_ev = lg[lg['path'].str.contains('compan', case=False, na=False)
             & ~lg['path'].str.contains(AUTO, case=False, regex=True)]
print('기업 정보 경로 상위 8'); print(comp_ev['path'].value_counts().head(8))
# 위 경로가 기업 정보 화면이 맞는지 확인한 뒤 아래 결과를 본다

comp_t = (comp_ev[['user_uuid','timestamp']].rename(columns={'timestamp':'t_comp'}).sort_values('t_comp'))
tmp = ap_t.dropna(subset=['t_det']).sort_values('t_start')
tmp = pd.merge_asof(tmp, comp_t, by='user_uuid', left_on='t_start', right_on='t_comp',
                    direction='backward', tolerance=pd.Timedelta('30min'))
tmp['기업정보 확인'] = tmp['t_comp'].notna() & (tmp['t_comp'] >= tmp['t_det'] - pd.Timedelta('5min'))
r1 = tmp.groupby('group')['기업정보 확인'].mean().mul(100).round(1).reindex(['G0','G1','A','B','C'])
print('① 지원 직전 기업 정보를 본 비율(%)'); display(r1.to_frame('비율'))

# ── ② 작성 중 멈춤: B vs 다른 그룹 (사용자 단위) ──
sets = {
    'B':             seg.loc[seg['group']=='B', 'user_uuid'],
    'G1-공고 탐색성': g1.loc[g1['view_score']==1, 'user_uuid'],
    'A':             seg.loc[seg['group']=='A', 'user_uuid'],
    'C':             seg.loc[seg['group']=='C', 'user_uuid'],
}
r2 = pd.concat({k: user_level_stall(v)['비율'] for k, v in sets.items()}, axis=1)
print('② 첫 STEP 2 이후 STEP 3 도달 (%)'); display(r2)

# ── ③ 지원 시간대 ──
a2['시간대'] = pd.cut(a2['t'].dt.tz_convert('Asia/Seoul').dt.hour, [-1,5,11,17,23],
                     labels=['새벽(0~5시)','오전(6~11시)','오후(12~17시)','저녁(18~23시)'])
tt = a2.merge(seg, on='user_uuid')
r3 = pd.crosstab(tt['시간대'], tt['group'], normalize='columns').mul(100).round(1)
print('③ 지원 시간대 분포(%)'); display(r3[['G0','G1','A','B','C']])

기업 정보 경로 상위 8
path
companies/company_id                   630069
companies/company_id/jobs              441571
api/search/companies                   195420
api/companies/id/member_list           155265
api/companies/id/ad_stat/progress      112814
api/companies/company_id                82623
companies                               62594
companies/company_id/jobs/job_title     50909
Name: count, dtype: int64
① 지원 직전 기업 정보를 본 비율(%)


,비율
group,
G0,60.7
G1,56.3
A,43.7
B,47.4
C,48.7


② 첫 STEP 2 이후 STEP 3 도달 (%)


,B,G1-공고 탐색성,A,C
결과,,,,
2시간 내 STEP 3 도달,79.1,71.4,82.7,86.0
2시간 이후 도달,18.3,17.4,16.7,13.5
STEP 3 도달 안 함,2.6,11.3,0.7,0.6


③ 지원 시간대 분포(%)


group,G0,G1,A,B,C
시간대,,,,,
새벽(0~5시),9.9,10.4,10.9,9.5,12.6
오전(6~11시),14.0,14.8,15.8,13.9,15.4
오후(12~17시),43.3,41.0,42.6,42.6,40.4
저녁(18~23시),32.8,33.8,30.7,34.0,31.5


## B 분석 결과

### 인사이트
- 작성 중 멈춤은 B의 문제가 아니다. 첫 STEP 2 이후 STEP 3에 끝내 도달하지 않은 비율이 B 2.6%로, 같은 빠른 판단 신호만 가진 G1(공고 탐색성) 11.3%보다 훨씬 낮다.
- 지원 시간대는 5개 그룹이 거의 같다. (오후 41~43%, 저녁 31~34%) 시간대는 B의 근거가 되지 못한다.
- 지원 직전 기업 정보 확인 비율은 B 47.4%로 G0(60.7%), G1(56.3%)보다 낮고, A(43.7%)·C(48.7%)와 비슷하다. 지원이 빠르거나 많은 그룹일수록 낮은 공통 경향이다.

### 결론
B만의 마찰은 아직 찾지 못했다. 기업 정보 확인은 B만의 특징이 아니고, 지표에 노이즈가 섞여 있어 확정하기 어렵다.

### 한계
- 기업 정보 경로에 기업 검색(`api/search/companies`), 멤버 목록, 광고 통계처럼 페이지를 열 때 자동으로 호출될 수 있는 경로가 섞여 있었다.
- 지원 완료 기록과 로그의 연결은 약 90%만 된다.

In [234]:
# ① 기업 정보 확인 (정제: 기업 페이지·기업 공고 목록만)
clean_paths = ['companies/company_id', 'api/companies/company_id', 'companies/company_id/jobs']
comp_t2 = (lg[lg['path'].isin(clean_paths)][['user_uuid','timestamp']]
           .rename(columns={'timestamp':'t_comp'}).sort_values('t_comp'))

tmp = ap_t.dropna(subset=['t_det']).sort_values('t_start')
tmp = pd.merge_asof(tmp, comp_t2, by='user_uuid', left_on='t_start', right_on='t_comp',
                    direction='backward', tolerance=pd.Timedelta('30min'))
tmp['기업정보 확인'] = tmp['t_comp'].notna() & (tmp['t_comp'] >= tmp['t_det'] - pd.Timedelta('5min'))
r1 = tmp.groupby('group')['기업정보 확인'].mean().mul(100).round(1).reindex(['G0','G1','A','B','C'])
print('① 지원 직전 기업 정보를 본 비율(%) - 정제'); display(r1.to_frame('비율'))

# ② 상세 페이지 체류 시간 (자동 호출 제외, 다음 행동까지 시간, 30분 이내)
ev = (lg[~lg['path'].str.contains(AUTO, case=False, regex=True)]
      .sort_values(['user_uuid','timestamp']).reset_index(drop=True))
ev['next_t'] = ev.groupby('user_uuid')['timestamp'].shift(-1)
d = ev[ev['path'] == 'api/jobs/job_title'].copy()
d['sec'] = (d['next_t'] - d['timestamp']).dt.total_seconds()
d = d[d['sec'].between(0, 1800)].merge(seg, on='user_uuid')

r2 = d.groupby('group')['sec'].agg(
    건수='size',
    **{'중앙값(초)': 'median'},
    **{'10초 이내(%)': lambda s: (s <= 10).mean() * 100},
    **{'60초 이상(%)': lambda s: (s >= 60).mean() * 100}).round(1).loc[['G0','G1','A','B','C']]
print('② 상세 페이지 체류 시간'); display(r2)

① 지원 직전 기업 정보를 본 비율(%) - 정제


,비율
group,
G0,49.0
G1,47.7
A,35.9
B,41.3
C,41.3


② 상세 페이지 체류 시간


,건수,중앙값(초),10초 이내(%),60초 이상(%)
group,,,,
G0,529087,11.6,45.6,11.4
G1,263155,11.4,45.9,10.4
A,21007,11.8,44.5,10.9
B,59649,11.2,46.4,9.5
C,29609,12.7,42.8,10.4


## B 추가 분석 결과

### 인사이트
- 지원 직전 기업 정보를 본 비율은 B 41.3%로 G0(49.0%), G1(47.7%)보다 낮다. 다만 C(41.3%)와 같고 A(35.9%)보다 높아, 지원이 많은 그룹의 공통 경향이다.
- 상세 페이지 체류 시간은 모든 그룹이 거의 같다. (중앙값 11~13초, 10초 이내 43~46%) B가 더 빨리 훑어보는 것은 아니다.
- 앞서 본 것처럼 작성 중 멈춤(도달 안 함 2.6%)과 지원 시간대에서도 B만의 특징은 없었다.

### 결론
B는 공고를 보는 속도와 방식은 다른 그룹과 같고, "조회 후 바로 지원한다"는 정의된 특징만 다르다. 데이터에서 찾은 유일한 단서는 지원 직전에 기업 정보를 보는 비율(41.3%)이 낮다는 점이지만, 차이가 작고 B에만 해당하지도 않는다.

### 한계
- 기업 정보 확인 비율은 지원 완료 기록과 로그의 연결(약 90%)에 의존한다.
- 기업 정보를 안 본다는 사실이 지원의 질이 낮다는 뜻은 아니다. 결과 데이터가 없다.

In [235]:
# ── 준비: 지원한 공고의 특징 ──
def to_bool(s):
    return s.astype(str).str.lower().isin(['1','true','t','y','yes'])

jf = job_df.drop_duplicates('job_uuid')[['job_uuid','cdate_kst','allow_remote','can_show_salary']].copy()
jf['posted'] = pd.to_datetime(jf['cdate_kst'], errors='coerce', utc=True)
jf['연봉공개'] = to_bool(jf['can_show_salary'])
jf['원격가능'] = to_bool(jf['allow_remote'])

a3 = fix(app_df)[['user_uuid','application_uuid','job_uuid','cdate']].copy()
a3['t'] = pd.to_datetime(a3['cdate'], errors='coerce', utc=True)
pa = (a3.dropna(subset=['t']).merge(jf[['job_uuid','posted','연봉공개','원격가능']], on='job_uuid', how='left')
         .merge(seg, on='user_uuid'))
pa['경과일'] = (pa['t'] - pa['posted']).dt.total_seconds() / 86400
pa.loc[pa['경과일'] < 0, '경과일'] = np.nan

# ── ① 그룹별 지원한 공고의 특징 ──
r1 = pa.groupby('group').agg(
    **{'연봉 공개 공고 비율(%)': ('연봉공개', lambda s: s.mean()*100)},
    **{'원격 가능 공고 비율(%)': ('원격가능', lambda s: s.mean()*100)},
    **{'등록 후 경과일 중앙값': ('경과일', 'median')},
    **{'등록 24시간 내 지원(%)': ('경과일', lambda s: (s <= 1).mean()*100)},
    **{'등록 3일 내 지원(%)': ('경과일', lambda s: (s <= 3).mean()*100)},
).round(1).loc[['G0','G1','A','B','C']]
r1.loc['(전체 공고)'] = [jf['연봉공개'].mean()*100, jf['원격가능'].mean()*100, np.nan, np.nan, np.nan]
print('① 지원한 공고의 특징'); display(r1.round(1))

# ── ② B 세부 유형 ──
bs = final_user_scores[final_user_scores[seg_col].astype(str).str.startswith('B_')].copy()
bs['user_uuid'] = bs['user_uuid'].astype(str)
bs['세부'] = np.select(
    [bs['diversity_score'] > 0, bs['concentration_score'] > 0, bs['volume_score'] > 0],
    ['다양성+탐색', '집중성+탐색', '대량성+탐색'], default='기타')
sub_map = bs[['user_uuid','세부']]

n_app_u = pa.groupby('user_uuid').size().rename('지원수')
view_to = ap_t.merge(sub_map, on='user_uuid').groupby('세부')['조회→지원시작(분)'].median().rename('조회→지원시작 중앙값(분)')
comp_ok = tmp.merge(sub_map, on='user_uuid').groupby('세부')['기업정보 확인'].mean().mul(100).rename('기업정보 확인(%)')
elapsed = pa.merge(sub_map, on='user_uuid').groupby('세부')['경과일'].median().rename('등록 후 경과일')
size = sub_map.groupby('세부').size().rename('인원')
napp = sub_map.merge(n_app_u, on='user_uuid', how='left').groupby('세부')['지원수'].median().rename('지원수 중앙값')

print('② B 세부 유형'); display(pd.concat([size, napp, view_to, comp_ok, elapsed], axis=1).round(1))

① 지원한 공고의 특징


,연봉 공개 공고 비율(%),원격 가능 공고 비율(%),등록 후 경과일 중앙값,등록 24시간 내 지원(%),등록 3일 내 지원(%)
group,,,,,
G0,38.4,21.6,51.0,5.1,10.8
G1,34.8,20.4,48.6,5.3,11.1
A,35.2,22.1,42.8,6.7,14.2
B,34.0,20.8,47.9,6.5,12.4
C,33.8,23.3,34.2,8.9,16.7
(전체 공고),60.2,12.3,NaN,NaN,NaN


② B 세부 유형


,인원,지원수 중앙값,조회→지원시작 중앙값(분),기업정보 확인(%),등록 후 경과일
세부,,,,,
다양성+탐색,281,8.0,1.7,39.1,42.8
대량성+탐색,97,39.0,2.4,38.9,48.1
집중성+탐색,114,9.0,3.3,57.3,59.2


## B 추가 분석 결과 (지원한 공고 특징, 세부 유형)

### 인사이트
- 지원한 공고의 특징은 B만의 것이 아니다. 연봉 공개 공고 비율은 모든 그룹이 34~38%로, 전체 공고(60.2%)보다 낮다. 등록 후 24시간 내 지원도 B 6.5%로 A(6.7%), G1(5.3%)과 비슷하다.
- B 안에서는 유형 간 차이가 크다.
  - 다양성+탐색(281명, B의 57%): 조회→지원 시작 중앙값 1.7분으로 가장 빠르고, 지원 전 기업 정보를 본 비율이 39.1%로 낮다.
  - 대량성+탐색(97명): 지원 중앙값 39건, 기업 정보 확인 38.9%.
  - 집중성+탐색(114명): 기업 정보 확인 57.3%로 오히려 G0(49.0%)보다 높고, 조회→지원도 3.3분으로 가장 느리다.
- 앞서 B 전체에서 본 낮은 기업 정보 확인(41.3%)은 다양성+탐색과 대량성+탐색(B의 77%)이 이끌고 있다.

### 결론
B의 중심은 **여러 직무 분야를 빠르게 지원하면서(다양성+탐색), 지원 직전에 기업 정보를 가장 덜 보는 사용자**다. 기업 정보 확인이 낮다는 가설은 이 유형에서 더 뚜렷하다. 다만 차이는 G0 대비 약 10%p로 크지 않다.

### 한계
- 집중성+탐색(114명), 대량성+탐색(97명)은 인원이 적어 참고용이다.
- 같은 사용자의 지원이 여러 건 들어가 있어 비율의 신뢰구간은 좁게 나오지 않는다.
- 기업 정보를 안 본다는 사실이 지원의 질이 낮다는 뜻은 아니다. 결과 데이터가 없다.

## B 개선안: 탐색 후 빠른 지원형(492명)

### 한눈에 보기
B는 공고를 보고 빠르게 지원하는 행동에 지원 행동 신호가 하나 더해진 그룹이다. 그중 57%(281명)는 여러 직무 분야를 탐색하며 빠르게 지원하고, 지원 직전에 기업 정보를 가장 덜 확인한다. 지원 직전에 기업 핵심 정보를 한눈에 보게 하는 것을 제안한다.

### 무엇을 발견했나
- B는 다양성+탐색(281명), 집중성+탐색(114명), 대량성+탐색(97명)으로 이루어진다.
- 지원 직전 기업 정보를 본 비율은 B 41.3%로 G0(49.0%)보다 낮다. 특히 다양성+탐색은 39.1%이고, 조회 후 지원 시작까지 중앙값 1.7분으로 가장 빠르다.
- 집중성+탐색은 기업 정보 확인 57.3%로 오히려 높아, 낮은 확인율은 B 전체의 특징이 아니라 유형별로 다르다.
- 작성 중 멈춤(첫 STEP 2 이후 STEP 3에 끝내 도달하지 않은 비율 2.6%), 상세 페이지 체류 시간(중앙값 11.2초), 지원 시간대는 다른 그룹과 차이가 없었다.

### 제안
**지원 직전 기업 한눈에 보기**: 지원을 시작하기 직전 단계에서 기업의 핵심 정보(규모·업력·근무조건 등)를 한 줄로 보여 주고, 확인한 뒤 지원하게 한다.
- 이유: 여러 직무를 빠르게 훑으며 지원하는 사용자는 기업을 하나씩 찾아볼 시간이 부족하므로, 지원 결정 순간에 필요한 정보를 한 번에 주면 도움이 될 것으로 가정한다.

### 효과 확인 방법
기업 한눈에 보기를 본 사용자와 보지 않은 사용자의 지원 완료율, 지원 취소·철회 비율, 지원 직전 기업 정보 확인 비율을 비교한다. (A/B 테스트)

### 한계
- 기업 정보 확인율 차이는 G0 대비 약 10%p로 작고, A와 C에서도 비슷한 경향이 있다.
- 집중성+탐색(114명), 대량성+탐색(97명)은 인원이 적어 참고용이다.
- 기업 정보를 안 본다는 것이 지원의 질이 낮다는 뜻은 아니다. 합격 등 결과 데이터가 없다.
- 관찰에서 나온 가설이며 실험으로 검증이 필요하다.

# C 개선안 도출

In [236]:
# 지원 세션 구성 (지원 간 간격 30분 초과 시 새 세션)
s = ap_t.sort_values(['user_uuid','t_apply']).copy()
s = s.merge(tmp[['application_uuid','기업정보 확인']], on='application_uuid', how='left')
s['기업정보 확인'] = s['기업정보 확인'].astype(float)

gap = s.groupby('user_uuid')['t_apply'].diff().dt.total_seconds() / 60
s['새세션'] = gap.isna() | (gap > 30)
s['세션'] = s.groupby('user_uuid')['새세션'].cumsum()
s['세션내순번'] = s.groupby(['user_uuid','세션']).cumcount() + 1

sess = s.groupby(['user_uuid','세션']).agg(
    group=('group','first'), 지원수=('application_uuid','size'),
    시작=('t_start','min'), 끝=('t_apply','max'))
sess['길이(분)'] = (sess['끝'] - sess['시작']).dt.total_seconds() / 60

# ① 세션 구조
g = sess.groupby('group')
r1 = pd.DataFrame({
    '세션 수': g.size(),
    '세션당 지원 수 중앙값': g['지원수'].median(),
    '세션당 지원 수 75%': g['지원수'].quantile(.75),
    '세션당 지원 수 90%': g['지원수'].quantile(.9),
    '세션 길이 중앙값(분)': g['길이(분)'].median(),
    '5건 이상 세션 비율(%)': g['지원수'].apply(lambda x: (x >= 5).mean()*100),
}).round(1).loc[['G0','G1','A','B','C']]
print('① 지원 세션 구조'); display(r1)

# ② 세션 안에서의 변화
s['세션내 구간'] = pd.cut(s['세션내순번'], [0,1,5,10,10**6], labels=['1번째','2~5번째','6~10번째','11번째+'])
x = s[s['group'].isin(['A','B','C'])]
col = '조회→지원시작(분)'
print('② 세션 내 조회→지원 시작 시간 중앙값(분)')
display(x.groupby(['세션내 구간','group'], observed=True)[col].median().unstack().round(1))
print('② 세션 내 기업 정보 확인 비율(%)')
display((x.groupby(['세션내 구간','group'], observed=True)['기업정보 확인'].mean()*100).unstack().round(1))
print('건수')
display(x.groupby(['세션내 구간','group'], observed=True).size().unstack())

① 지원 세션 구조


,세션 수,세션당 지원 수 중앙값,세션당 지원 수 75%,세션당 지원 수 90%,세션 길이 중앙값(분),5건 이상 세션 비율(%)
group,,,,,,
G0,19328,1.0,1.0,2.0,3.4,1.8
G1,13842,1.0,2.0,3.0,2.2,4.9
A,2215,2.0,4.0,9.0,6.2,23.3
B,4097,1.0,2.0,5.0,2.4,10.7
C,2639,2.0,5.0,10.0,5.4,28.5


② 세션 내 조회→지원 시작 시간 중앙값(분)


group,A,B,C
세션내 구간,,,
1번째,2.7,1.7,1.5
2~5번째,4.7,2.5,2.5
6~10번째,6.4,3.1,2.8
11번째+,6.5,4.0,3.6


② 세션 내 기업 정보 확인 비율(%)


group,A,B,C
세션내 구간,,,
1번째,30.7,33.8,28.1
2~5번째,38.4,47.1,41.2
6~10번째,40.9,44.7,47.1
11번째+,31.5,39.3,48.1


건수


group,A,B,C
세션내 구간,,,
1번째,2215,4097,2639
2~5번째,3249,4056,4509
6~10번째,1475,922,2238
11번째+,1369,388,2725


## C 지원 세션 분석

### 인사이트
- C는 지원 세션이 가장 길게 이어진다. 세션당 지원 수 90% 지점이 10건(A 9건, B 5건)이고, 5건 이상 세션 비율은 C 28.5%, A 23.3%, B 10.7%다. 세션 내 11번째 이후 지원도 C가 2,725건으로 A(1,369건)의 2배다.
- 세션 뒤로 갈수록 정보를 덜 보는 가속 패턴은 없었다. C의 조회→지원 시작 시간은 1.5분 → 2.5분 → 2.8분 → 3.6분으로 오히려 길어지고, 기업 정보 확인은 28.1% → 41.2% → 47.1% → 48.1%로 늘어난다.
- 세션의 첫 지원에서 기업 정보 확인이 가장 낮은 것은 A(30.7%), B(33.8%), C(28.1%) 모두 같다.

### 결론
C의 특징은 판단 속도가 아니라 **연속 지원이 길게 이어진다는 점**이다. 세션 안에서 정보 확인이 줄어드는 패턴은 없어서, A(대량 지원일의 낮은 조회)와 같은 방향의 제안은 C에 맞지 않는다.

### 다음 분석
C가 지원한 공고에 어떻게 도달하는지(검색·목록·다른 공고·기타) 본다. 신규 공고 반응(등록 24시간 내 지원 8.9%)이 높은 점과 연결해, 공고를 찾는 방식에서 C만의 특징이 있는지 확인한다.

### 한계
- 세션은 로그와 연결된 지원(약 90%)만으로 만들었다.
- 세션 안 순번별 비교는 앞부분(첫 지원)과 뒷부분의 사용자 구성이 달라 단순 비교가 어렵다.

In [237]:
# ① 지원한 공고의 상세 조회 직전 행동 (진입 경로)
ev2 = ev.copy()
ev2['prev_path'] = ev2.groupby('user_uuid')['path'].shift(1)
dd = (ev2[ev2['path'] == 'api/jobs/job_title'][['user_uuid','timestamp','prev_path']]
      .drop_duplicates(['user_uuid','timestamp']).rename(columns={'timestamp':'t_det'}))

ent = ap_t[['user_uuid','t_det','group']].dropna().merge(dd, on=['user_uuid','t_det'], how='left')
p = ent['prev_path'].fillna('')
ent['진입'] = np.select(
    [p.str.contains('api/search', case=False),
     p == 'jobs',
     p.str.contains('jobs/id/id_title|api/jobs/job_title', case=False),
     p.str.contains('apply/step', case=False),
     p.str.contains('compan', case=False),
     p.str.contains('applications|job_offer|resume|profile', case=False),
     p == ''],
    ['검색', '공고 목록', '다른 공고 상세', '지원 직후 이어짐', '기업 페이지', '내 정보', '없음(세션 시작)'],
    default='기타')
r1 = pd.crosstab(ent['진입'], ent['group'], normalize='columns').mul(100).round(1)
print('① 지원한 공고에 도달한 직전 행동(%)'); display(r1[['G0','G1','A','B','C']])
print('기타 상위 경로'); print(ent.loc[ent['진입']=='기타','prev_path'].value_counts().head(8))

# ② 검색 요청 URL에 들어 있는 옵션(쿼리 키) 확인
q = fix(log_df)[['user_uuid','URL','path']]
q = q[q['path'].astype(str).str.contains('api/search', case=False, na=False)].copy()
keys = q['URL'].astype(str).str.extract(r'\?(.*)$')[0].dropna().str.split('&').explode().str.split('=').str[0]
print('\n② 검색 URL 쿼리 키 상위 15'); print(keys.value_counts().head(15))

① 지원한 공고에 도달한 직전 행동(%)


group,G0,G1,A,B,C
진입,,,,,
검색,2.3,2.1,1.4,1.3,1.2
공고 목록,26.7,23.8,20.5,22.9,18.7
기업 페이지,4.8,5.2,3.8,4.4,4.9
기타,2.9,3.6,2.8,4.5,6.0
내 정보,0.6,0.7,1.0,0.5,0.7
다른 공고 상세,60.1,60.5,61.5,57.6,50.2
없음(세션 시작),0.4,0.3,0.3,0.5,0.4
지원 직후 이어짐,2.2,3.8,8.7,8.3,17.8


기타 상위 경로
prev_path
api/jobs/id/template_oneclick    732
api/jobs/id/bookmark             611
api/users/id/template            351
api/recommend_specialty          282
api/users/id/experience/form      87
@user_id                          68
api/jobs/user_filter/id           49
api/timeline                      35
Name: count, dtype: int64

② 검색 URL 쿼리 키 상위 15
0
name         1115735
_             710106
keywords      143607
q             106224
exclude        34671
type           31461
job             1697
location        1142
specialty        577
school           468
                 460
                 212
status           109
D                 69
rel               52
Name: count, dtype: int64


## C 지원한 공고의 진입 경로 분석

### 인사이트
- 지원한 공고의 상세 조회 직전 행동은 대부분 "다른 공고 상세"(50~60%)와 "공고 목록"(19~27%)이다. 검색에서 바로 들어오는 비율은 모든 그룹에서 1~2%로 낮다.
- C만 두드러지는 것은 **지원 직후 이어짐**이다. 지원한 공고를 보게 된 직전 행동이 지원 단계였던 비율이 C 17.8%로, A 8.7%, B 8.3%, G1 3.8%, G0 2.2%보다 높다. C는 지원을 끝낸 바로 다음 공고로 이어서 가는 흐름이 가장 강하다.
- 반대로 "다른 공고 상세"에서 이어진 비율은 C가 50.2%로 가장 낮다.
- 검색 URL에는 정렬·필터 옵션이 거의 쓰이지 않는다. (키 이름 `name`, `keywords`, `q` 위주이고 `location`·`job`·`specialty`는 수백~천 건대)

### 결론
C는 지원 완료 직후의 흐름이 다음 지원으로 이어지는 연속 지원형이다. 앞서 본 C의 특징(5건 이상 세션 28.5%, 지원 직후 30분 내 이어짐 67.0%, 등록 24시간 내 지원 8.9%로 신규 공고에 가장 빠르게 반응)과 같은 방향이다.

### 한계
- 지원 직후 이어짐이 어떤 화면에서 이어지는지(추천 영역인지, 목록인지)는 로그만으로는 알 수 없다.
- 진입 경로는 지원 기록과 연결된 지원(약 90%)만 반영한다.
- C는 171명으로 인원이 적다.

In [238]:
ent2 = ent.merge(pa[['user_uuid','t','경과일']].rename(columns={'t':'t_apply'}),
                 left_on='user_uuid', right_on='user_uuid', how='left') if False else None
# 지원(ap_t)과 공고 경과일(pa)을 application_uuid로 연결
fresh = ap_t[['application_uuid','user_uuid','t_det','group']].merge(
    pa[['application_uuid','경과일']], on='application_uuid', how='left'
).merge(dd, on=['user_uuid','t_det'], how='left')
fresh['직전'] = np.where(fresh['prev_path'].fillna('').str.contains('apply/step', case=False),
                         '지원 직후 이어짐', '그 외')
x = fresh[fresh['group'].isin(['A','B','C'])]
print('등록 후 경과일 중앙값')
display(x.groupby(['직전','group'])['경과일'].median().unstack().round(1))
print('등록 3일 내 비율(%)')
display((x.groupby(['직전','group'])['경과일'].apply(lambda s: (s<=3).mean()*100)).unstack().round(1))

등록 후 경과일 중앙값


group,A,B,C
직전,,,
그 외,37.5,46.9,35.2
지원 직후 이어짐,50.2,55.8,27.0


등록 3일 내 비율(%)


group,A,B,C
직전,,,
그 외,17.4,13.0,16.5
지원 직후 이어짐,4.5,6.4,18.8


## C 지원 직후 이어진 공고의 신선도 분석

### 인사이트
- C가 지원 직후 이어서 지원한 공고는 등록 후 경과일 중앙값이 27.0일로, 그 외 경로(35.2일)보다 짧다. 등록 3일 내 비율도 18.8%로 그 외(16.5%)보다 높다.
- 같은 지원 직후 경로가 A와 B에서는 반대로 오래된 공고로 이어진다. (경과일 A 50.2일, B 55.8일 / 등록 3일 내 A 4.5%, B 6.4%)
- 즉 지원 직후 흐름에서 C만 신규 공고 쪽으로 움직인다. 지원 직후 등록 3일 내 공고에 지원한 비율은 C가 A의 약 4배, B의 약 3배다.

### 결론
C는 지원을 끝낸 직후 이어서 신규 공고를 찾아 지원하는 연속 지원형이다. 앞서 본 지원 세션이 길고(5건 이상 세션 28.5%), 지원 직후 다음 공고로 이어지는 비율이 가장 높은 점과 같은 방향이다.

### 한계
- C 내부에서 지원 직후 경로와 그 외 경로의 차이는 크지 않다. (경과일 8일, 3일 내 비율 2.3%p)
- 중앙값 27일은 절대적으로 신규라고 하기 어렵고, A·B와의 상대 비교로만 해석한다.
- C는 171명이라 표본이 작고, 지원 직후 화면이 추천 영역인지 목록인지는 로그로 알 수 없다.

## C 개선안: 다수 공고 빠른 지원형(171명)

### 한눈에 보기
C는 조회 후 빠르게 지원하는 행동과 여러 공고에 지원하는 행동이 함께 나타나는 그룹이다. 지원을 끝낸 직후 바로 다음 공고로 이어서 지원하는 흐름이 가장 강하고, 그 흐름에서 신규 공고를 찾는 경향이 있다. 지원 직후에 최근 등록된 공고를 이어서 보여 주는 것을 제안한다.

### 무엇을 발견했나
- C는 지원 세션이 길다. 한 세션에 5건 이상 지원하는 비율이 28.5%(A 23.3%, B 10.7%)이고, 세션당 지원 수 90% 지점이 10건이다.
- 지원 직후 30분 안에 다른 공고 상세로 이어지는 비율이 67.0%로 가장 높고, 지원한 공고의 17.8%는 직전 행동이 지원 단계였다. (A 8.7%, B 8.3%, G0 2.2%)
- 그 지원 직후 흐름에서 C는 등록 3일 내 공고에 지원한 비율이 18.8%로, A(4.5%), B(6.4%)보다 높다.
- 작성 시간(6번째 이후 0.3분), 세션 안 정보 확인 감소 같은 가속 패턴은 없었다.

### 제안
**지원 직후 신규 공고 이어주기**: 지원을 완료한 직후 화면에서, 같은 직무의 최근 등록된 공고를 이어서 보여 준다.
- 이유: C는 지원 직후 다음 공고로 이어가는 흐름이 가장 강하고 그 흐름에서 신규 공고에 반응하므로, 그 순간에 신규 공고를 놓으면 연속 지원이 더 쉬워질 것으로 가정한다.

### 효과 확인 방법
지원 직후 신규 공고를 본 사용자와 보지 않은 사용자의 세션당 지원 수, 지원 직후 30분 내 이어짐 비율을 비교한다. (A/B 테스트)

### 한계
- C 내부의 지원 직후 경로와 그 외 경로의 신선도 차이는 크지 않다. (경과일 27.0일 대 35.2일, 3일 내 18.8% 대 16.5%)
- 지원 직후 화면이 어떤 영역인지는 로그로 알 수 없고, 유사 공고 추천은 이미 제공되는 서비스가 있을 수 있다.
- C는 171명으로 표본이 작고, 관찰에서 나온 가설이다. 실험으로 검증이 필요하다.

# 그룹별 개선안 (Project 2: 채용 플랫폼 지원자 행동 유형화)

## 한눈에 보기

| 그룹 | 개선 지점 | 제안 |
|---|---|---|
| G0 (신호 없음, 63%) | 지원 시작 전 | 후보 비교판 |
| G1 (단일 신호, 28.6%) | 지원서 작성 중 | 멈춘 단계로 바로 복귀 |
| A 다수 공고 지원형 (151명) | 대량 지원 흐름 | 일괄 확인 후 지원 |
| B 탐색 후 빠른 지원형 (492명) | 지원 직전 | 기업 한눈에 보기 |
| C 다수 공고 빠른 지원형 (171명) | 지원 직후 | 신규 공고 이어주기 |

---

## G0 개선안: 일반 사용자(신호 없는 63%)

### 한눈에 보기
공고를 보고도 지원으로 이어지지 않는 일반 사용자가 많다. 이들은 지원하지 않은 뒤 대부분 다른 공고를 계속 둘러본다. 여러 공고를 한 화면에서 비교할 수 있게 하는 것을 제안한다.

### 무엇을 발견했나
- 공고를 본 사용자 중 34%가 이후 지원을 시작하지 않았다.
- 공고를 같은 횟수만큼 봐도 다른 그룹에 비해 G0는 지원을 시작하는 비율이 가장 낮았다.
- 공고를 3번 넘게 봐도 지원 시작 비율은 약 16~17%에서 더 오르지 않았다.
- 지원으로 이어지지 않은 공고 조회(474,877건) 뒤의 다음 행동은 다른 공고 페이지 이동 63.1%, 검색·목록 복귀 10.6%였다. 합치면 약 74%가 바로 다른 공고를 계속 탐색했다.
- 공고를 북마크로 남기는 경우는 0.7%, 바로 서비스를 나가는 경우는 12.7%였다.

### 제안
**후보 비교판**: 최근 본 공고 2~3개를 나란히 놓고, 연봉·근무지·필수 자격의 차이를 한 화면에서 비교해 준다.
- 이유: G0는 공고를 계속 넘겨 보지만 기억에 남기거나 저장하지 않는다. 비교 기준을 화면에서 대신 정리해 주면 지원 판단이 쉬워질 것으로 가정한다.

### 효과 확인 방법
비교판을 본 사용자와 보지 않은 사용자의 지원 시작 비율을 비교한다. (A/B 테스트)

### 한계
- 로그에 공고 ID가 가려져 있어, "다른 공고 페이지"가 실제로 다른 공고인지는 확정할 수 없다.
- 다음 행동 비율은 다른 그룹도 비슷하므로 G0만의 특징은 아니다. G0의 낮은 지원 시작률과 함께 나타나는 패턴으로 해석한다.
- 이 데이터로는 지원하지 않은 이유(조건이 안 맞아서인지, 구경만 했는지)를 알 수 없다. 제안은 실험으로 검증해야 한다.

---

## G1 개선안: 단일 신호 사용자(28.6%)

### 한눈에 보기
G1의 90%는 공고를 보고 빠르게 지원을 시작하는 사용자(공고 탐색성 71.4%)와 여러 직무에 지원하는 사용자(다양성 19.1%)다. 이들은 지원서를 쓰다 멈추면 돌아와도 이어쓰지 않고 처음부터 다시 시작한다. 멈춘 단계로 바로 돌아오게 하는 것을 제안한다.

### 무엇을 발견했나
- 두 신호 모두 지원서 작성 중간(STEP 2→3)에서 이어지지 않은 비율이 가장 컸다. (공고 탐색성 19.2%, 다양성 16.0%)
- 첫 STEP 2 이후 2시간 안에 STEP 3까지 간 사용자는 71~73%였고, 나머지 약 28%는 첫 지원서를 STEP 2에서 끝내지 못했다.
- 나중에 STEP 3에 도달한 사용자의 96~99%는 그 사이에 STEP 1을 다시 시작했다. 이어서 쓴 게 아니라 처음부터 다시 쓴 것으로 보인다.

### 제안
**멈춘 단계로 바로 복귀**: 지원서를 쓰다 멈춘 사용자가 다시 들어오면, 처음부터가 아니라 마지막으로 멈춘 단계(STEP 2)로 바로 이동시킨다.
- 이유: 멈춘 뒤 돌아와도 STEP 1부터 다시 시작하는 사용자가 대부분이므로, 재진입 경로를 줄이면 작성 완료가 늘어날 것으로 가정한다.

### 효과 확인 방법
멈춘 단계로 복귀한 사용자와 그렇지 않은 사용자의 지원 완료 비율(STEP 4 도달), 재작성(STEP 1 재시작) 비율을 비교한다. (A/B 테스트)

### 한계
- 공고 ID가 가려져 있어, 다시 시작한 지원서가 같은 공고인지 다른 공고인지는 확정할 수 없다.
- STEP 1 재기록은 같은 지원서에서 앞 단계로 돌아간 경우도 포함할 수 있다.
- 대량성·집중성·북마크 탐색성은 인원이 적어(72~134명) 이번 제안의 근거에서 제외했다.
- 제안의 효과는 이 분석으로 알 수 없고, 실험으로 검증이 필요하다.

---

## A 개선안: 다수 공고 지원형(151명)

### 한눈에 보기
A는 지원 행동 신호(대량성·집중성·다양성)가 복합적으로 나타나지만, 조회 후 바로 지원하는 공고 탐색성은 없는 그룹이다. 하루에 지원을 몰아서 할수록 공고 1건당 확인(상세 조회)이 줄어든다. 여러 공고를 한 화면에서 확인하고 지원하는 방식을 제안한다.

### 무엇을 발견했나
- A는 지원 건수 중앙값이 38건, 활동일당 5.7건이다. (정의와 겹치는 지표라서 특징 설명에만 쓴다.)
- 작성 시간은 지원 횟수가 늘수록 줄었다. (1번째 1.9분 → 21번째 이후 0.5분) 반복 입력 부담은 보이지 않았다.
- 하루 지원이 많을수록 지원 1건당 상세 조회가 줄었다. (1건 4.0 → 5~9건 1.13 → 10건 이상 0.67) 모든 구간에서 B, C보다 조회가 적었다.
- 기업·마감·경력 측면에서는 다른 그룹과 뚜렷한 차이가 없었다.

### 제안
**일괄 확인 후 지원**: 관심 공고 여러 개를 한 화면에 모아 마감일·자격·근무조건을 한 번에 확인하고 지원하게 한다.
- 이유: 하루에 많이 지원할수록 공고별로 상세 페이지를 열어 보는 횟수가 줄므로, 한 화면에서 확인할 수 있게 하면 대량 지원 흐름에 맞을 것으로 가정한다.

### 효과 확인 방법
일괄 확인 화면을 쓴 사용자와 쓰지 않은 사용자의 지원 완료율과 지원 후 취소·철회 비율을 비교한다. (A/B 테스트)

### 한계
- 지원당 조회가 1 미만인 경우는 전날 조회·북마크·로그에 안 잡히는 경로일 수 있다.
- 조회가 적다는 것이 지원의 질이 낮다는 뜻은 아니다. 합격 등 결과 데이터가 없다.
- 인원이 151명으로 적고, 분석은 관찰에서 나온 가설이다. 실험으로 검증이 필요하다.

---

## B 개선안: 탐색 후 빠른 지원형(492명)

### 한눈에 보기
B는 공고를 보고 빠르게 지원하는 행동에 지원 행동 신호가 하나 더해진 그룹이다. 그중 57%(281명)는 여러 직무 분야를 탐색하며 빠르게 지원하고, 지원 직전에 기업 정보를 가장 덜 확인한다. 지원 직전에 기업 핵심 정보를 한눈에 보게 하는 것을 제안한다.

### 무엇을 발견했나
- B는 다양성+탐색(281명), 집중성+탐색(114명), 대량성+탐색(97명)으로 이루어진다.
- 지원 직전 기업 정보를 본 비율은 B 41.3%로 G0(49.0%)보다 낮다. 특히 다양성+탐색은 39.1%이고, 조회 후 지원 시작까지 중앙값 1.7분으로 가장 빠르다.
- 집중성+탐색은 기업 정보 확인 57.3%로 오히려 높아, 낮은 확인율은 B 전체의 특징이 아니라 유형별로 다르다.
- 작성 중 멈춤(첫 STEP 2 이후 STEP 3에 끝내 도달하지 않은 비율 2.6%), 상세 페이지 체류 시간(중앙값 11.2초), 지원 시간대는 다른 그룹과 차이가 없었다.

### 제안
**지원 직전 기업 한눈에 보기**: 지원을 시작하기 직전 단계에서 기업의 핵심 정보(규모·업력·근무조건 등)를 한 줄로 보여 주고, 확인한 뒤 지원하게 한다.
- 이유: 여러 직무를 빠르게 훑으며 지원하는 사용자는 기업을 하나씩 찾아볼 시간이 부족하므로, 지원 결정 순간에 필요한 정보를 한 번에 주면 도움이 될 것으로 가정한다.

### 효과 확인 방법
기업 한눈에 보기를 본 사용자와 보지 않은 사용자의 지원 완료율, 지원 취소·철회 비율, 지원 직전 기업 정보 확인 비율을 비교한다. (A/B 테스트)

### 한계
- 기업 정보 확인율 차이는 G0 대비 약 10%p로 작고, A와 C에서도 비슷한 경향이 있다.
- 집중성+탐색(114명), 대량성+탐색(97명)은 인원이 적어 참고용이다.
- 기업 정보를 안 본다는 것이 지원의 질이 낮다는 뜻은 아니다. 합격 등 결과 데이터가 없다.
- 관찰에서 나온 가설이며 실험으로 검증이 필요하다.

---

## C 개선안: 다수 공고 빠른 지원형(171명)

### 한눈에 보기
C는 조회 후 빠르게 지원하는 행동과 여러 공고에 지원하는 행동이 함께 나타나는 그룹이다. 지원을 끝낸 직후 바로 다음 공고로 이어서 지원하는 흐름이 가장 강하고, 그 흐름에서 신규 공고를 찾는 경향이 있다. 지원 직후에 최근 등록된 공고를 이어서 보여 주는 것을 제안한다.

### 무엇을 발견했나
- C는 지원 세션이 길다. 한 세션에 5건 이상 지원하는 비율이 28.5%(A 23.3%, B 10.7%)이고, 세션당 지원 수 90% 지점이 10건이다.
- 지원 직후 30분 안에 다른 공고 상세로 이어지는 비율이 67.0%로 가장 높고, 지원한 공고의 17.8%는 직전 행동이 지원 단계였다. (A 8.7%, B 8.3%, G0 2.2%)
- 그 지원 직후 흐름에서 C는 등록 3일 내 공고에 지원한 비율이 18.8%로, A(4.5%), B(6.4%)보다 높다.
- 작성 시간(6번째 이후 0.3분), 세션 안 정보 확인 감소 같은 가속 패턴은 없었다.

### 제안
**지원 직후 신규 공고 이어주기**: 지원을 완료한 직후 화면에서, 같은 직무의 최근 등록된 공고를 이어서 보여 준다.
- 이유: C는 지원 직후 다음 공고로 이어가는 흐름이 가장 강하고 그 흐름에서 신규 공고에 반응하므로, 그 순간에 신규 공고를 놓으면 연속 지원이 더 쉬워질 것으로 가정한다.

### 효과 확인 방법
지원 직후 신규 공고를 본 사용자와 보지 않은 사용자의 세션당 지원 수, 지원 직후 30분 내 이어짐 비율을 비교한다. (A/B 테스트)

### 한계
- C 내부의 지원 직후 경로와 그 외 경로의 신선도 차이는 크지 않다. (경과일 27.0일 대 35.2일, 3일 내 18.8% 대 16.5%)
- 지원 직후 화면이 어떤 영역인지는 로그로 알 수 없고, 유사 공고 추천은 이미 제공되는 서비스가 있을 수 있다.
- C는 171명으로 표본이 작고, 관찰에서 나온 가설이다. 실험으로 검증이 필요하다.

---

## 공통 한계
- 이 분석은 관찰 데이터라서 상관관계이지 인과관계가 아니다. 제안의 효과는 A/B 테스트로 검증해야 한다.
- 로그에 공고 ID가 가려져 있어 공고 단위 분석은 하지 못했다.
- "이어지지 않음"은 120분 안에 다음 단계 기록이 없다는 뜻이고, 포기 여부는 알 수 없다.
- 합격 등 지원 결과 데이터가 없어, 지원의 질이나 만족도는 판단하지 못했다.